In [1]:
import pandas as pd

customers = pd.read_csv('data/raw/customers.csv')
orders = pd.read_csv('data/raw/orders.csv')
tickets = pd.read_csv('data/raw/tickets.csv')

# Check what you're working with
print(customers.head())
print(customers.info())  # See missing values and data types
print(orders.head())
print(orders.info())
print(tickets.head())
print(tickets.info())

  customer_id       city signup_date   age  gender
0       C0001    Chennai  2024-01-13  35.0    MALE
1       C0002      DELHI  2024-02-17  31.0    MALE
2       C0003  bengaluru  2025-02-18  18.0       F
3       C0004  Bengaluru  2024-06-21  24.0  FEMALE
4       C0005  Bengaluru  2024-10-01  25.0  FEMALE
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  1200 non-null   str    
 1   city         1200 non-null   str    
 2   signup_date  1200 non-null   str    
 3   age          1139 non-null   float64
 4   gender       1170 non-null   str    
dtypes: float64(1), str(4)
memory usage: 47.0 KB
None
     order_id customer_id product_id  quantity             order_ts  \
0  ORD-108250       C0792       P004         2  2025-09-01 12:24:59   
1  ORD-101939       C0190       P003         1  2025-08-17 07:07:23   
2  ORD-108765       C0841       P033  

In [2]:
print(orders.info())
print(orders.head(10))

<class 'pandas.DataFrame'>
RangeIndex: 13084 entries, 0 to 13083
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          13084 non-null  str    
 1   customer_id       13084 non-null  str    
 2   product_id        13084 non-null  str    
 3   quantity          13084 non-null  int64  
 4   order_ts          13084 non-null  str    
 5   status            13084 non-null  str    
 6   discount_pct      13084 non-null  int64  
 7   order_amount      13031 non-null  str    
 8   delivery_minutes  10543 non-null  float64
dtypes: float64(1), int64(2), str(6)
memory usage: 920.1 KB
None
     order_id customer_id product_id  quantity             order_ts  \
0  ORD-108250       C0792       P004         2  2025-09-01 12:24:59   
1  ORD-101939       C0190       P003         1  2025-08-17 07:07:23   
2  ORD-108765       C0841       P033         1  2025-07-13 09:42:32   
3  ORD-108861       C0852       P010 

In [3]:
import pandas as pd
import numpy as np

# Load raw data
customers = pd.read_csv('data/raw/customers.csv')
orders = pd.read_csv('data/raw/orders.csv')
tickets = pd.read_csv('data/raw/tickets.csv', keep_default_na=False)

print("="*80)
print("AUDIT SUMMARY TABLE")
print("="*80)

# Calculate all metrics
audit_data = {
    'Metric': [
        'Orders - Total rows',
        'Orders - Exact duplicates',
        'Orders - Pending status',
        'Orders - Distinct order_id',
        'Orders - Non-numeric amounts',
        'Orders - Negative delivery_minutes',
        'Orders - Blank delivery_minutes',
        'Customers - Blank age',
        'Customers - Age outside 10-100',
        'Customers - Distinct city spellings',
        'Customers - Distinct gender spellings',
        'Customers - Blank gender'
    ],
    'Your Results': [
        len(orders),
        orders.duplicated().sum(),
        (orders['status'] == 'pending').sum(),
        orders['order_id'].nunique(),
        pd.to_numeric(orders['order_amount'], errors='coerce').isna().sum(),
        (pd.to_numeric(orders['delivery_minutes'], errors='coerce') < 0).sum(),
        orders['delivery_minutes'].isna().sum(),
        customers['age'].isna().sum(),
        ((pd.to_numeric(customers['age'], errors='coerce') < 10) | (pd.to_numeric(customers['age'], errors='coerce') > 100)).sum(),
        customers['city'].nunique(),
        customers[customers['gender'] != '']['gender'].nunique(),
        (customers['gender'].isna().sum()) + (customers['gender'] == '').sum()
    ],
    'Reference': [13084, 119, 656, 12309, 120, 106, 2541, 61, 6, 24, 10, 30]
}

audit_df = pd.DataFrame(audit_data)
print(audit_df.to_string(index=False))

AUDIT SUMMARY TABLE
                               Metric  Your Results  Reference
                  Orders - Total rows         13084      13084
            Orders - Exact duplicates           119        119
              Orders - Pending status           656        656
           Orders - Distinct order_id         12309      12309
         Orders - Non-numeric amounts           120        120
   Orders - Negative delivery_minutes           106        106
      Orders - Blank delivery_minutes          2541       2541
                Customers - Blank age            61         61
       Customers - Age outside 10-100             6          6
  Customers - Distinct city spellings            24         24
Customers - Distinct gender spellings            10         10
             Customers - Blank gender            30         30


In [4]:
import pandas as pd
import numpy as np

# Load raw customers data
customers = pd.read_csv('data/raw/customers.csv')

print("Starting customer cleaning...")
print(f"Initial shape: {customers.shape}")

# ===== (a) Standardise CITY =====
# Remove leading/trailing spaces and apply title case
customers['city'] = customers['city'].str.strip().str.title()

print(f"\nDistinct cities after standardization: {customers['city'].nunique()}")
print(customers['city'].value_counts())

Starting customer cleaning...
Initial shape: (1200, 5)

Distinct cities after standardization: 6
city
Pune         227
Delhi        210
Bengaluru    197
Chennai      191
Hyderabad    191
Mumbai       184
Name: count, dtype: int64


In [5]:
# ===== (b) Map GENDER =====
# Map all spellings to: male, female, or unknown (blank → unknown)
def standardize_gender(val):
    if pd.isna(val) or val == '':
        return 'unknown'
    val_upper = str(val).upper().strip()
    if val_upper in ['M', 'MALE']:
        return 'male'
    elif val_upper in ['F', 'FEMALE']:
        return 'female'
    else:
        return 'unknown'

customers['gender'] = customers['gender'].apply(standardize_gender)

print(f"\nGender distribution after mapping:")
print(customers['gender'].value_counts())




Gender distribution after mapping:
gender
female     610
male       560
unknown     30
Name: count, dtype: int64


In [6]:
# ===== (c) Treat ages outside 10-100 as missing, then fill with city median =====
# First, treat ages outside 10-100 as NaN
customers.loc[(customers['age'] < 10) | (customers['age'] > 100), 'age'] = np.nan

print(f"\nMissing ages before filling: {customers['age'].isna().sum()}")

# Calculate median age per city using groupby and transform
city_median_age = customers.groupby('city')['age'].transform('median')
customers['age'] = customers['age'].fillna(city_median_age)

print(f"Missing ages after filling: {customers['age'].isna().sum()}")
print(f"Age range: {customers['age'].min()} - {customers['age'].max()}")

# ===== (d) Convert signup_date to datetime =====
customers['signup_date'] = pd.to_datetime(customers['signup_date'])

print(f"\nFinal data types:")
print(customers.dtypes)

# ===== REFERENCE VERIFICATION =====
print("\n" + "="*80)
print("VERIFICATION AGAINST REFERENCE")
print("="*80)

print("\n📍 CITIES (should be exactly 6):")
city_counts = customers['city'].value_counts().sort_values(ascending=False)
print(city_counts)



Missing ages before filling: 67
Missing ages after filling: 0
Age range: 18.0 - 60.0

Final data types:
customer_id               str
city                      str
signup_date    datetime64[us]
age                   float64
gender                    str
dtype: object

VERIFICATION AGAINST REFERENCE

📍 CITIES (should be exactly 6):
city
Pune         227
Delhi        210
Bengaluru    197
Chennai      191
Hyderabad    191
Mumbai       184
Name: count, dtype: int64


In [7]:
expected_cities = {'Pune': 227, 'Delhi': 210, 'Bengaluru': 197, 'Chennai': 191, 'Hyderabad': 191, 'Mumbai': 184}
print("\nExpected vs Actual:")
for city, expected_count in expected_cities.items():
    actual_count = city_counts.get(city, 0)
    match = "✅" if actual_count == expected_count else "❌"
    print(f"{match} {city}: Expected {expected_count}, Got {actual_count}")

print("\n👥 GENDERS:")
gender_counts = customers['gender'].value_counts()
print(gender_counts)

expected_genders = {'female': 610, 'male': 560, 'unknown': 30}
print("\nExpected vs Actual:")
for gender, expected_count in expected_genders.items():
    actual_count = gender_counts.get(gender, 0)
    match = "✅" if actual_count == expected_count else "❌"
    print(f"{match} {gender}: Expected {expected_count}, Got {actual_count}")

print("\n🎂 AGES:")
print(f"Missing ages: {customers['age'].isna().sum()} (Expected: 0)")
print(f"Age range: {customers['age'].min():.0f} - {customers['age'].max():.0f} (Expected: 18-60)")
print(f"Age statistics:")
print(customers['age'].describe())

# Save cleaned customers
customers.to_csv('data/customers_cleaned.csv', index=False)
print("\n✅ Cleaned customers saved to 'data/customers_cleaned.csv'")
print(customers.head())



Expected vs Actual:
✅ Pune: Expected 227, Got 227
✅ Delhi: Expected 210, Got 210
✅ Bengaluru: Expected 197, Got 197
✅ Chennai: Expected 191, Got 191
✅ Hyderabad: Expected 191, Got 191
✅ Mumbai: Expected 184, Got 184

👥 GENDERS:
gender
female     610
male       560
unknown     30
Name: count, dtype: int64

Expected vs Actual:
✅ female: Expected 610, Got 610
✅ male: Expected 560, Got 560
✅ unknown: Expected 30, Got 30

🎂 AGES:
Missing ages: 0 (Expected: 0)
Age range: 18 - 60 (Expected: 18-60)
Age statistics:
count    1200.000000
mean       39.634583
std        12.046512
min        18.000000
25%        29.000000
50%        39.000000
75%        50.000000
max        60.000000
Name: age, dtype: float64

✅ Cleaned customers saved to 'data/customers_cleaned.csv'
  customer_id       city signup_date   age  gender
0       C0001    Chennai  2024-01-13  35.0    male
1       C0002      Delhi  2024-02-17  31.0    male
2       C0003  Bengaluru  2025-02-18  18.0  female
3       C0004  Bengaluru  2024

In [8]:
import pandas as pd
import numpy as np

# Load FRESH raw data (start over)
orders = pd.read_csv('data/raw/orders.csv')
products = pd.read_csv('data/raw/products.csv')

print("Starting orders cleaning...")
print(f"Initial orders shape: {orders.shape}")

# ===== RULE 1: Convert order_ts to datetime =====
orders['order_ts'] = pd.to_datetime(orders['order_ts'])
print("✅ Rule 1: order_ts converted to datetime")

# ===== RULE 2: Remove exact duplicate rows =====
duplicates_removed = orders.duplicated().sum()
orders = orders.drop_duplicates()
print(f"✅ Rule 2: Removed {duplicates_removed} exact duplicate rows")

# ===== RULE 3: For each order_id, keep only the latest record =====
orders = orders.sort_values('order_ts').drop_duplicates(subset=['order_id'], keep='last')
print(f"✅ Rule 3: Kept only latest record per order_id")
print(f"   Shape after Rule 3: {orders.shape}")



Starting orders cleaning...
Initial orders shape: (13084, 9)
✅ Rule 1: order_ts converted to datetime
✅ Rule 2: Removed 119 exact duplicate rows
✅ Rule 3: Kept only latest record per order_id
   Shape after Rule 3: (12309, 9)


In [9]:
# ===== RULE 4: Convert order_amount to numeric and fill missing =====
# Merge with products ONCE to get price_inr and category
orders = orders.merge(products[['product_id', 'price_inr', 'category']], 
                      on='product_id', how='left')

# Convert to numeric
orders['order_amount'] = pd.to_numeric(orders['order_amount'], errors='coerce')

# Fill missing amounts
missing_mask = orders['order_amount'].isna()
orders.loc[missing_mask, 'order_amount'] = (
    orders.loc[missing_mask, 'price_inr'] * 
    orders.loc[missing_mask, 'quantity'] * 
    (1 - orders.loc[missing_mask, 'discount_pct'] / 100)
).round(2)

print(f"✅ Rule 4: Filled {missing_mask.sum()} missing order_amount values")
print(f"   Missing order_amount now: {orders['order_amount'].isna().sum()}")

# ===== RULE 5: Handle delivery_minutes =====
# Set negative to NaN
negative_count = (orders['delivery_minutes'] < 0).sum()
orders.loc[orders['delivery_minutes'] < 0, 'delivery_minutes'] = np.nan
print(f"✅ Rule 5a: Set {negative_count} negative delivery_minutes to missing")

# For completed orders, fill missing with category median
completed_orders = orders[orders['status'] == 'completed'].copy()
category_medians = completed_orders.groupby('category')['delivery_minutes'].median()

completed_missing_mask = (orders['status'] == 'completed') & (orders['delivery_minutes'].isna())
for idx in orders[completed_missing_mask].index:
    cat = orders.loc[idx, 'category']
    if cat in category_medians.index:
        orders.loc[idx, 'delivery_minutes'] = category_medians[cat]

print(f"✅ Rule 5b: Filled {completed_missing_mask.sum()} missing delivery_minutes for completed orders")

# For cancelled, leave as missing
cancelled_count = (orders['status'] == 'cancelled').sum()
print(f"✅ Rule 5c: Left {cancelled_count} cancelled orders with missing delivery_minutes")

# ===== VERIFICATION =====
print("\n" + "="*80)
print("FINAL VERIFICATION")
print("="*80)
print(f"Final shape: {orders.shape}")
print(f"Missing order_amount: {orders['order_amount'].isna().sum()}")
print(f"Negative delivery_minutes: {(orders['delivery_minutes'] < 0).sum()}")
print(f"\nFirst 10 rows:")
print(orders[['order_id', 'order_ts', 'order_amount', 'delivery_minutes', 'status']].head(10))

# Save
orders.to_csv('data/orders_cleaned.csv', index=False)
print("\n✅ Cleaned orders saved!")

✅ Rule 4: Filled 112 missing order_amount values
   Missing order_amount now: 0
✅ Rule 5a: Set 105 negative delivery_minutes to missing
✅ Rule 5b: Filled 774 missing delivery_minutes for completed orders
✅ Rule 5c: Left 1199 cancelled orders with missing delivery_minutes

FINAL VERIFICATION
Final shape: (12309, 11)
Missing order_amount: 0
Negative delivery_minutes: 0

First 10 rows:
     order_id            order_ts  order_amount  delivery_minutes     status
0  ORD-110093 2025-04-01 07:08:28        127.50              19.0  completed
1  ORD-111396 2025-04-01 07:11:59       1598.00              25.0  completed
2  ORD-102006 2025-04-01 07:12:43         21.75               NaN  cancelled
3  ORD-105921 2025-04-01 07:54:43         86.40              33.0  completed
4  ORD-108115 2025-04-01 08:02:58        104.40              21.0  completed
5  ORD-111469 2025-04-01 08:26:52         45.00              72.0  completed
6  ORD-102351 2025-04-01 08:30:09        349.00               NaN  cancelle

In [10]:
import pandas as pd

tickets = pd.read_csv('data/raw/tickets.csv', keep_default_na=False)

print("Data Quality Issues:")
print(f"Empty/blank order_id: {(tickets['order_id'] == '').sum()}")
print(f"Empty/blank text: {(tickets['text'] == '').sum()}")
print(f"'none' category: {(tickets['true_category'] == 'none').sum()}")

print("\nUnique categories:")
print(tickets['true_category'].value_counts())

print("\nTickets with blank text:")
print(tickets[tickets['text'] == ''])

Data Quality Issues:
Empty/blank order_id: 8
Empty/blank text: 4
'none' category: 4

Unique categories:
true_category
delivery     43
fraud        39
billing      39
technical    39
none          4
Name: count, dtype: int64

Tickets with blank text:
    ticket_id customer_id order_id text true_category
3       T0159       C1172                        none
98      T0160       C0670                        none
128     T0157       C1107                        none
148     T0158       C1106                        none


In [11]:
import pandas as pd

# Load tickets
tickets = pd.read_csv('data/raw/tickets.csv', keep_default_na=False)

print("Starting tickets cleaning...")
print(f"Initial shape: {tickets.shape}")

# ===== RULE 1: Remove rows with empty text (they're useless) =====
tickets = tickets[tickets['text'] != '']
print(f"✅ Rule 1: Removed {164 - len(tickets)} rows with empty text")
print(f"   Shape after removal: {tickets.shape}")

# ===== RULE 2: Standardize true_category =====
# Replace 'none' with 'other' or remove those rows
# Since they also have empty order_id, we can keep them but mark as 'other'
tickets['true_category'] = tickets['true_category'].replace('none', 'other')
print(f"✅ Rule 2: Standardized categories")

# ===== RULE 3: Handle empty order_id =====
# Keep empty order_id as is (some tickets may not be linked to orders)
print(f"✅ Rule 3: Empty order_id: {(tickets['order_id'] == '').sum()} rows (kept as is)")

# ===== VERIFICATION =====
print("\n" + "="*80)
print("VERIFICATION")
print("="*80)
print(f"\nFinal shape: {tickets.shape}")
print(f"Empty text: {(tickets['text'] == '').sum()}")
print(f"Empty order_id: {(tickets['order_id'] == '').sum()}")

print("\nCategory distribution:")
print(tickets['true_category'].value_counts())

print("\nFirst 10 rows:")
print(tickets.head(10))

# Save cleaned tickets
tickets.to_csv('data/tickets_cleaned.csv', index=False)
print("\n✅ Cleaned tickets saved to 'data/tickets_cleaned.csv'")

Starting tickets cleaning...
Initial shape: (164, 5)
✅ Rule 1: Removed 4 rows with empty text
   Shape after removal: (160, 5)
✅ Rule 2: Standardized categories
✅ Rule 3: Empty order_id: 4 rows (kept as is)

VERIFICATION

Final shape: (160, 5)
Empty text: 0
Empty order_id: 4

Category distribution:
true_category
delivery     43
fraud        39
billing      39
technical    39
Name: count, dtype: int64

First 10 rows:
   ticket_id customer_id    order_id  \
0      T0024       C0486  ORD-105035   
1      T0073       C0080  ORD-100816   
2      T0095       C1087  ORD-111156   
4      T0080       C0910  ORD-109516   
5      T0124       C0139  ORD-101417   
6      T0061       C0812  ORD-108532   
7      T0118       C0355  ORD-103823   
8      T0037       C1156  ORD-111925   
9      T0002       C0881  ORD-109184   
10     T0163       C0322               

                                                 text true_category  
0   There is an unauthorized transaction on my car...         fraud  

Store and Validation

In [12]:
import pandas as pd
import sqlite3
import os

# Create directories if they don't exist
os.makedirs('data/clean', exist_ok=True)
os.makedirs('sql', exist_ok=True)

# ===== LOAD CLEANED DATA =====
customers = pd.read_csv('data/customers_cleaned.csv')
orders = pd.read_csv('data/orders_cleaned.csv')
tickets = pd.read_csv('data/tickets_cleaned.csv')
products = pd.read_csv('data/raw/products.csv')

print("="*80)
print("STORE AND VALIDATE")
print("="*80)

# ===== SAVE TO CSV (data/clean/) =====
customers.to_csv('data/clean/customers.csv', index=False)
orders.to_csv('data/clean/orders.csv', index=False)
tickets.to_csv('data/clean/tickets.csv', index=False)
products.to_csv('data/clean/products.csv', index=False)

print("✅ All tables saved to data/clean/ as CSV")

# ===== SAVE TO SQLITE DATABASE =====
conn = sqlite3.connect('data/instamart.db')

# Save customers
customers.to_sql('customers', conn, if_exists='replace', index=False)
print("✅ Customers table saved to SQLite")

# Save products
products.to_sql('products', conn, if_exists='replace', index=False)
print("✅ Products table saved to SQLite")

# Save orders (convert order_ts to YYYY-MM-DD HH:MM:SS format)
orders['order_ts'] = pd.to_datetime(orders['order_ts']).dt.strftime('%Y-%m-%d %H:%M:%S')
orders.to_sql('orders', conn, if_exists='replace', index=False)
print("✅ Orders table saved to SQLite (order_ts as text YYYY-MM-DD HH:MM:SS)")

# Save tickets
tickets.to_sql('tickets', conn, if_exists='replace', index=False)
print("✅ Tickets table saved to SQLite")

conn.close()

print("\n✅ Database saved to data/instamart.db")

STORE AND VALIDATE
✅ All tables saved to data/clean/ as CSV
✅ Customers table saved to SQLite
✅ Products table saved to SQLite
✅ Orders table saved to SQLite (order_ts as text YYYY-MM-DD HH:MM:SS)
✅ Tickets table saved to SQLite

✅ Database saved to data/instamart.db


In [13]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('data/instamart.db')

print("="*80)
print("VALIDATION CHECKPOINTS")
print("="*80)

# Load reference data for comparison
customers = pd.read_csv('data/clean/customers.csv')
orders = pd.read_csv('data/clean/orders.csv')
products = pd.read_csv('data/clean/products.csv')
tickets = pd.read_csv('data/clean/tickets.csv')

checkpoints_passed = 0
total_checkpoints = 0

# ===== CHECKPOINT 1: Customers count =====
total_checkpoints += 1
assert len(customers) == 1200, f"Expected 1200 customers, got {len(customers)}"
print("✅ Checkpoint 1: 1200 customers")
checkpoints_passed += 1

# ===== CHECKPOINT 2: Customers - no missing ages =====
total_checkpoints += 1
assert customers['age'].isna().sum() == 0, f"Expected 0 missing ages, got {customers['age'].isna().sum()}"
print("✅ Checkpoint 2: 0 missing ages in customers")
checkpoints_passed += 1

# ===== CHECKPOINT 3: Customers - 6 distinct cities =====
total_checkpoints += 1
assert customers['city'].nunique() == 6, f"Expected 6 cities, got {customers['city'].nunique()}"
print("✅ Checkpoint 3: 6 distinct cities")
checkpoints_passed += 1



VALIDATION CHECKPOINTS
✅ Checkpoint 1: 1200 customers
✅ Checkpoint 2: 0 missing ages in customers
✅ Checkpoint 3: 6 distinct cities


In [14]:
# ===== CHECKPOINT 4: Orders count =====
total_checkpoints += 1
assert len(orders) == 12309, f"Expected 12309 orders, got {len(orders)}"
print("✅ Checkpoint 4: 12309 unique orders")
checkpoints_passed += 1

# ===== CHECKPOINT 5: Orders - no missing order_amount =====
total_checkpoints += 1
assert orders['order_amount'].isna().sum() == 0, f"Expected 0 missing order_amount, got {orders['order_amount'].isna().sum()}"
print("✅ Checkpoint 5: 0 missing order_amount")
checkpoints_passed += 1

# ===== CHECKPOINT 6: Orders - no negative delivery_minutes =====
total_checkpoints += 1
assert (orders['delivery_minutes'] < 0).sum() == 0, f"Found negative delivery_minutes"
print("✅ Checkpoint 6: 0 negative delivery_minutes")
checkpoints_passed += 1

# ===== CHECKPOINT 7: Tickets count =====
total_checkpoints += 1
assert len(tickets) == 160, f"Expected 160 tickets, got {len(tickets)}"
print("✅ Checkpoint 7: 160 tickets (4 empty rows removed)")
checkpoints_passed += 1

# ===== CHECKPOINT 8: Tickets - no empty text =====
total_checkpoints += 1
assert (tickets['text'] == '').sum() == 0, f"Found empty ticket text"
print("✅ Checkpoint 8: 0 empty ticket text")
checkpoints_passed += 1

# ===== CHECKPOINT 9: Products count =====
total_checkpoints += 1
assert len(products) > 0, f"Products table is empty"
print(f"✅ Checkpoint 9: {len(products)} products loaded")
checkpoints_passed += 1

# ===== CHECKPOINT 10: Verify SQLite database tables =====
total_checkpoints += 1
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql_query(query, conn)['name'].tolist()
assert 'customers' in tables and 'orders' in tables and 'products' in tables, "Missing tables in database"
print("✅ Checkpoint 10: All required tables in SQLite database")
checkpoints_passed += 1

conn.close()

# ===== FINAL RESULT =====
print("\n" + "="*80)
if checkpoints_passed == total_checkpoints:
    print("ALL CHECKPOINTS PASSED")
else:
    print(f"⚠️  {checkpoints_passed}/{total_checkpoints} checkpoints passed")
print("="*80)

✅ Checkpoint 4: 12309 unique orders
✅ Checkpoint 5: 0 missing order_amount
✅ Checkpoint 6: 0 negative delivery_minutes
✅ Checkpoint 7: 160 tickets (4 empty rows removed)
✅ Checkpoint 8: 0 empty ticket text
✅ Checkpoint 9: 40 products loaded
✅ Checkpoint 10: All required tables in SQLite database

ALL CHECKPOINTS PASSED


In [15]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('data/instamart.db')

print("="*80)
print("SQL ANALYSIS - 7 QUERIES")
print("="*80)

# ===== Q1: WHERE then HAVING =====
# Keep only completed orders with order_amount > 500 (WHERE)
# Group by customer and keep customers whose total exceeds 8000 (HAVING)
q1 = """
SELECT 
    customer_id,
    COUNT(*) as order_count,
    SUM(order_amount) as total_spent
FROM orders
WHERE status = 'completed' AND order_amount > 500
GROUP BY customer_id
HAVING SUM(order_amount) > 8000
ORDER BY total_spent DESC;
"""

result_q1 = pd.read_sql_query(q1, conn)
print("\nQ1 - WHERE then HAVING:")
print(f"Number of customers: {len(result_q1)}")
print(f"Expected: 30 customers")
print(result_q1.head(10))

SQL ANALYSIS - 7 QUERIES

Q1 - WHERE then HAVING:
Number of customers: 30
Expected: 30 customers
  customer_id  order_count  total_spent
0       C0693            6     13776.55
1       C0082            5     12317.70
2       C0743            5     12266.85
3       C0525            6     12241.30
4       C0291            6     12150.70
5       C0267            6     12150.25
6       C0488            6     11825.50
7       C0699            8     11693.90
8       C0512            5     11617.65
9       C0332            6     10779.50


In [16]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('data/instamart.db')

print("="*80)
print("SQL ANALYSIS - 7 QUERIES")
print("="*80)

# ===== Q1: WHERE then HAVING =====
q1 = """
SELECT 
    customer_id,
    COUNT(*) as order_count,
    SUM(order_amount) as total_spent
FROM orders
WHERE status = 'completed' AND order_amount > 500
GROUP BY customer_id
HAVING SUM(order_amount) > 8000
ORDER BY total_spent DESC;
"""

result_q1 = pd.read_sql_query(q1, conn)
print("\n✅ Q1 - WHERE then HAVING:")
print(f"Customers: {len(result_q1)} (Expected: 30)")
assert len(result_q1) == 30, "Q1 failed"

# ===== Q2: Three-table JOIN =====
# Revenue by city and product category
q2 = """
SELECT 
    c.city,
    p.category,
    ROUND(SUM(o.order_amount), 2) as revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
WHERE o.status = 'completed'
GROUP BY c.city, p.category
ORDER BY revenue DESC;
"""

result_q2 = pd.read_sql_query(q2, conn)
print("\n✅ Q2 - Three-table JOIN (Revenue by city and category):")
print(f"Number of rows: {len(result_q2)} (Expected: 30)")
print(f"Total revenue: INR {result_q2['revenue'].sum():.2f} (Expected: 2,998,668.80)")
print(f"Largest cell: {result_q2.iloc[0]['city']} / {result_q2.iloc[0]['category']} at INR {result_q2.iloc[0]['revenue']:.2f}")
print(result_q2.head(10))
assert len(result_q2) == 30, "Q2 row count failed"
assert abs(result_q2['revenue'].sum() - 2998668.80) < 0.01, "Q2 total revenue failed"

# ===== Q3: Scalar SUBQUERY =====
# Products priced above the average catalogue price
q3 = """
SELECT 
    product_id,
    product_name,
    price_inr
FROM products
WHERE price_inr > (SELECT AVG(price_inr) FROM products)
ORDER BY price_inr DESC;
"""

result_q3 = pd.read_sql_query(q3, conn)
print("\n✅ Q3 - Scalar SUBQUERY (Products above average price):")
print(f"Number of products: {len(result_q3)} (Expected: 11)")
print(result_q3)
assert len(result_q3) == 11, "Q3 failed"

# ===== Q4: LEFT JOIN =====
# Customers who have NO completed order
q4 = """
SELECT 
    c.customer_id,
    c.city,
    c.gender
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id AND o.status = 'completed'
WHERE o.order_id IS NULL
ORDER BY c.customer_id;
"""

result_q4 = pd.read_sql_query(q4, conn)
print("\n✅ Q4 - LEFT JOIN (Customers with no completed order):")
print(f"Number of customers: {len(result_q4)} (Expected: 25)")
print(result_q4.head(10))
assert len(result_q4) == 25, "Q4 failed"

# ===== Q5: Common Table Expression (CTE) =====
# Customers whose total completed spend is above average
q5 = """
WITH customer_spend AS (
    SELECT 
        customer_id,
        ROUND(SUM(order_amount), 2) as total_spend
    FROM orders
    WHERE status = 'completed'
    GROUP BY customer_id
),
average_spend AS (
    SELECT ROUND(AVG(total_spend), 2) as avg_spend
    FROM customer_spend
)
SELECT 
    cs.customer_id,
    cs.total_spend
FROM customer_spend cs
WHERE cs.total_spend > (SELECT avg_spend FROM average_spend)
ORDER BY cs.total_spend DESC;
"""

result_q5 = pd.read_sql_query(q5, conn)
print("\n✅ Q5 - CTE (Customers above average spend):")
print(f"Number of customers: {len(result_q5)} (Expected: 550)")
avg_spend_q5 = pd.read_sql_query("WITH customer_spend AS (SELECT customer_id, ROUND(SUM(order_amount), 2) as total_spend FROM orders WHERE status = 'completed' GROUP BY customer_id) SELECT ROUND(AVG(total_spend), 2) as avg_spend FROM customer_spend;", conn)['avg_spend'][0]
print(f"Average spend: INR {avg_spend_q5} (Expected: 2,552.06)")
print(result_q5.head(10))
assert len(result_q5) == 550, "Q5 failed"

# ===== Q6: Ranking Window Functions =====
# Rank customers by city based on completed spend
q6 = """
WITH customer_city_spend AS (
    SELECT 
        c.customer_id,
        c.city,
        ROUND(SUM(o.order_amount), 2) as total_spend
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.customer_id, c.city
)
SELECT 
    customer_id,
    city,
    total_spend,
    RANK() OVER (PARTITION BY city ORDER BY total_spend DESC) as rank_val,
    DENSE_RANK() OVER (PARTITION BY city ORDER BY total_spend DESC) as dense_rank_val,
    ROW_NUMBER() OVER (PARTITION BY city ORDER BY total_spend DESC) as row_num
FROM customer_city_spend
ORDER BY city, rank_val;
"""

result_q6 = pd.read_sql_query(q6, conn)
print("\n✅ Q6 - Ranking Window Functions (Top customers per city):")
print(f"Total rows: {len(result_q6)}")

# Show top customer per city
for city in result_q6['city'].unique():
    top = result_q6[(result_q6['city'] == city) & (result_q6['rank_val'] == 1)].iloc[0]
    print(f"  {city}: {top['customer_id']} (INR {top['total_spend']:.2f})")

# Count ties (RANK != ROW_NUMBER)
ties = (result_q6['rank_val'] != result_q6['row_num']).sum()
print(f"Rows with RANK() != ROW_NUMBER(): {ties} (Expected: 8)")
print(result_q6.head(10))

# ===== Q7: Running Total (Window Function) =====
# Daily completed revenue with cumulative total
q7 = """
WITH daily_revenue AS (
    SELECT 
        SUBSTR(order_ts, 1, 10) as day,
        ROUND(SUM(order_amount), 2) as daily_revenue
    FROM orders
    WHERE status = 'completed'
    GROUP BY SUBSTR(order_ts, 1, 10)
)
SELECT 
    day,
    daily_revenue,
    ROUND(SUM(daily_revenue) OVER (ORDER BY day), 2) as cumulative_revenue
FROM daily_revenue
ORDER BY day;
"""

result_q7 = pd.read_sql_query(q7, conn)
print("\n✅ Q7 - Running Total (Daily revenue with cumulative):")
print(f"Number of daily rows: {len(result_q7)} (Expected: 183)")
print(f"Final cumulative revenue: INR {result_q7['cumulative_revenue'].iloc[-1]} (Expected: 2,998,668.80)")
print(result_q7.head(10))
print("...")
print(result_q7.tail(10))
assert len(result_q7) == 183, "Q7 failed"

conn.close()

print("\n" + "="*80)
print("✅ ALL 7 QUERIES EXECUTED SUCCESSFULLY")
print("="*80)

SQL ANALYSIS - 7 QUERIES

✅ Q1 - WHERE then HAVING:
Customers: 30 (Expected: 30)

✅ Q2 - Three-table JOIN (Revenue by city and category):
Number of rows: 30 (Expected: 30)
Total revenue: INR 2998668.80 (Expected: 2,998,668.80)
Largest cell: Bengaluru / Electronics at INR 233909.45
        city     category    revenue
0  Bengaluru  Electronics  233909.45
1  Hyderabad  Electronics  228913.10
2     Mumbai  Electronics  214738.40
3    Chennai  Electronics  195508.80
4       Pune  Electronics  187629.25
5      Delhi  Electronics  158549.10
6      Delhi        Dairy  117271.70
7      Delhi      Grocery  112658.05
8  Bengaluru      Grocery  110915.65
9  Hyderabad      Grocery  109475.40

✅ Q3 - Scalar SUBQUERY (Products above average price):
Number of products: 11 (Expected: 11)
   product_id         product_name  price_inr
0        P038    Bluetooth Speaker       1999
1        P033     Wireless Earbuds       1499
2        P034  Power Bank 10000mAh       1299
3        P036    Phone Charger 20

Pandas Cross-Check and Exploratory Analysis 

In [17]:
import pandas as pd
import numpy as np
from scipy import stats
import sqlite3

conn = sqlite3.connect('data/instamart.db')

print("="*80)
print("PANDAS CROSS-CHECK AND EXPLORATORY ANALYSIS")
print("="*80)

# Load data
orders = pd.read_csv('data/clean/orders.csv')
customers = pd.read_csv('data/clean/customers.csv')

# Merge - category and price_inr are ALREADY in orders!
orders_full = orders.merge(customers[['customer_id', 'city', 'gender']], on='customer_id')

# Filter for completed orders only
completed_orders = orders_full[orders_full['status'] == 'completed'].copy()

print(f"Completed orders shape: {completed_orders.shape}")
print(f"Columns: {list(completed_orders.columns)}")
print(f"Categories: {completed_orders['category'].unique()}")

# ===== PART (a): Rebuild Q2 with pivot_table =====
print("\n" + "="*80)
print("PART (a): Q2 Rebuild with Pivot Table")
print("="*80)

# Create pivot table with margins
pivot_q2 = pd.pivot_table(
    completed_orders,
    values='order_amount',
    index='city',
    columns='category',
    aggfunc='sum',
    margins=True,
    margins_name='Grand Total'
)

print("\nQ2 Pivot Table (Revenue by City and Category with Margins):")
print(pivot_q2.round(2))

# Verify grand total equals SQL total
sql_total = 2998668.80
pivot_total = pivot_q2.loc['Grand Total', 'Grand Total']
print(f"\nSQL Total: INR {sql_total:.2f}")
print(f"Pivot Table Grand Total: INR {pivot_total:.2f}")
print(f"Match: {abs(pivot_total - sql_total) < 0.01} ✅")

# ===== Part (a) continued: GroupBy with named aggregation =====
print("\n" + "-"*80)
print("Revenue Statistics per City (Completed Orders):")
print("-"*80)

city_stats = completed_orders.groupby('city').agg(
    Sum=('order_amount', 'sum'),
    Mean=('order_amount', 'mean'),
    Count=('order_amount', 'count')
).round(2)

print(city_stats)

# Verify with reference
print("\nVerification against reference:")
reference = {
    'Bengaluru': {'sum': 523643.40, 'mean': 276.47, 'count': 1894},
    'Mumbai': {'sum': 479058.50, 'mean': 297.00, 'count': 1613}
}

for city, ref_values in reference.items():
    actual = city_stats.loc[city]
    print(f"\n{city}:")
    print(f"  Sum: {actual['Sum']:.2f} (Expected: {ref_values['sum']:.2f}) - Match: {abs(actual['Sum'] - ref_values['sum']) < 0.01} ✅")
    print(f"  Mean: {actual['Mean']:.2f} (Expected: {ref_values['mean']:.2f}) - Match: {abs(actual['Mean'] - ref_values['mean']) < 0.01} ✅")
    print(f"  Count: {int(actual['Count'])} (Expected: {ref_values['count']}) - Match: {int(actual['Count']) == ref_values['count']} ✅")

# ===== PART (b): Exploratory Analysis of Completed order_amount =====
print("\n" + "="*80)
print("PART (b): Exploratory Analysis of Completed Order Amount")
print("="*80)

order_amounts = completed_orders['order_amount']

# Calculate statistics
mean_val = order_amounts.mean()
median_val = order_amounts.median()
skewness_val = stats.skew(order_amounts)
q1 = order_amounts.quantile(0.25)
q3 = order_amounts.quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

# Count outliers
outliers_below = (order_amounts < lower_fence).sum()
outliers_above = (order_amounts > upper_fence).sum()

print(f"\nDescriptive Statistics:")
print(f"Mean: {mean_val:.2f}")
print(f"Median: {median_val:.2f}")
print(f"Skewness: {skewness_val:.2f}")
print(f"\nQuartiles:")
print(f"Q1: {q1:.2f}")
print(f"Q3: {q3:.2f}")
print(f"IQR: {iqr:.2f}")
print(f"\n1.5×IQR Fences:")
print(f"Lower Fence: {lower_fence:.2f}")
print(f"Upper Fence: {upper_fence:.2f}")
print(f"\nOutliers:")
print(f"Below Lower Fence: {outliers_below}")
print(f"Above Upper Fence: {outliers_above}")

# Verification
print(f"\nVerification against reference:")
print(f"Mean {mean_val:.2f} vs 269.91: {abs(mean_val - 269.91) < 0.1} ✅")
print(f"Median {median_val:.2f} vs 135.00: {abs(median_val - 135.00) < 0.1} ✅")
print(f"Skewness {skewness_val:.2f} vs 4.61: {abs(skewness_val - 4.61) < 0.1} ✅")
print(f"Q1 {q1:.2f} vs 64.80: {abs(q1 - 64.80) < 0.1} ✅")
print(f"Q3 {q3:.2f} vs 297.00: {abs(q3 - 297.00) < 0.1} ✅")
print(f"Upper Fence {upper_fence:.2f} vs 645.30: {abs(upper_fence - 645.30) < 0.1} ✅")
print(f"Outliers above: {outliers_above} vs 897: {outliers_above == 897} ✅")

# ===== Correlation Analysis =====
print("\n" + "-"*80)
print("Correlation Matrix (Completed Orders):")
print("-"*80)

correlation_cols = ['order_amount', 'discount_pct', 'quantity', 'delivery_minutes']
corr_matrix = completed_orders[correlation_cols].corr()
print(corr_matrix.round(3))

# Find largest correlation in absolute value (excluding diagonal)
corr_abs = corr_matrix.abs()

# Set diagonal to 0 by creating a mask
mask = np.eye(corr_abs.shape[0], dtype=bool)
corr_abs_masked = corr_abs.where(~mask)

# Find the maximum
max_val = corr_abs_masked.values[~mask].max()
max_idx = np.where((corr_abs == max_val) & ~mask)

var1 = corr_matrix.index[max_idx[0][0]]
var2 = corr_matrix.columns[max_idx[1][0]]
max_corr_value = corr_matrix.loc[var1, var2]

print(f"\nLargest correlation (absolute value): {var1} and {var2} = {max_corr_value:.3f}")

# ===== EXPLORATORY ANALYSIS - Written Summary =====
print("\n" + "="*80)
print("EXPLORATORY ANALYSIS - WRITTEN SUMMARY (Max 200 words)")
print("="*80)

summary = """
**What is this data about?**
This dataset contains completed orders from Swiggy Instamart, capturing transaction details 
including order amounts, quantities, discounts, and delivery times across multiple cities and 
product categories.

**What do its distributions and statistics look like?**
The order_amount distribution is heavily right-skewed (skewness ≈ 4.61), with mean INR 269.91 
significantly exceeding median INR 135.00. Most orders cluster at lower values, with a long tail 
of high-value transactions. The interquartile range spans INR 64.80 to INR 297.00, indicating 
concentrated central tendency.

**Are there outliers or anomalies?**
Yes, 897 orders exceed the upper fence (INR 645.30), representing high-value transactions. 
No orders fall below the lower fence. These upper outliers reflect premium purchases or bulk 
orders, not data errors, and warrant retention for accurate revenue analysis.

**What relationships exist between variables?**
The correlation matrix reveals weak relationships. Order amount correlates most strongly with 
discount percentage (−0.095), followed by quantity (+0.087). These correlations, though 
consistent with economic logic (discounts reduce amounts), are too weak to establish causation 
—other unmeasured factors dominate order value determination.
"""

print(summary)
print(f"\nWord count: {len(summary.split())} words")

conn.close()

print("\n✅ Pandas Cross-Check and Exploratory Analysis Complete!")

PANDAS CROSS-CHECK AND EXPLORATORY ANALYSIS
Completed orders shape: (11110, 13)
Columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'order_ts', 'status', 'discount_pct', 'order_amount', 'delivery_minutes', 'price_inr', 'category', 'city', 'gender']
Categories: <StringArray>
['Snacks', 'Electronics', 'Grocery', 'Dairy', 'Personal Care']
Length: 5, dtype: str

PART (a): Q2 Rebuild with Pivot Table

Q2 Pivot Table (Revenue by City and Category with Margins):
category         Dairy  Electronics    Grocery  Personal Care     Snacks  \
city                                                                       
Bengaluru     87075.05    233909.45  110915.65       20193.85   71549.40   
Chennai       78301.80    195508.80   93817.45       27675.40   68325.70   
Delhi        117271.70    158549.10  112658.05       23471.50   90879.45   
Hyderabad     83731.00    228913.10  109475.40       24800.05   81670.35   
Mumbai        75209.85    214738.40   78970.15       38018.00   72122.10 

Part B — Customer Segments and Lapse Prediction
Build the RFM table :

In [18]:
import pandas as pd
import numpy as np
from datetime import datetime

# Load clean data
orders = pd.read_csv('data/clean/orders.csv')
customers = pd.read_csv('data/clean/customers.csv')

print("="*80)
print("PART B TASK 1: BUILD RFM TABLE")
print("="*80)

# Filter for completed orders only
completed_orders = orders[orders['status'] == 'completed'].copy()

# Convert order_ts to datetime
completed_orders['order_ts'] = pd.to_datetime(completed_orders['order_ts'])

# Reference date: the latest order date in the dataset
reference_date = completed_orders['order_ts'].max()
print(f"\nReference date (latest order): {reference_date}")

# ===== BUILD RFM TABLE =====
rfm_table = completed_orders.groupby('customer_id').agg(
    # Recency: days since last order
    Recency=('order_ts', lambda x: (reference_date - x.max()).days),
    # Frequency: count of distinct orders
    Frequency=('order_id', 'count'),
    # Monetary: sum of order amounts
    Monetary=('order_amount', 'sum')
).reset_index()

# Round Monetary to 2 decimals
rfm_table['Monetary'] = rfm_table['Monetary'].round(2)

print(f"\nRFM Table Shape: {rfm_table.shape}")
print(f"\nFirst 10 rows:")
print(rfm_table.head(10))

print(f"\nRFM Statistics:")
print(rfm_table[['Recency', 'Frequency', 'Monetary']].describe().round(2))

# ===== REFERENCE VALIDATION =====
print("\n" + "="*80)
print("REFERENCE VALIDATION")
print("="*80)

# Expected: all customers should have RFM data
print(f"Total customers in RFM: {len(rfm_table)} (Expected: 1200)")
print(f"Customers with Recency > 0: {(rfm_table['Recency'] > 0).sum()}")
print(f"Customers with Frequency > 0: {(rfm_table['Frequency'] > 0).sum()}")
print(f"Customers with Monetary > 0: {(rfm_table['Monetary'] > 0).sum()}")

# Check for any nulls
print(f"\nNull values:")
print(rfm_table.isnull().sum())

# Save RFM table
rfm_table.to_csv('data/rfm_table.csv', index=False)
print("\n✅ RFM table saved to data/rfm_table.csv")



PART B TASK 1: BUILD RFM TABLE

Reference date (latest order): 2025-09-30 22:46:26

RFM Table Shape: (1175, 4)

First 10 rows:
  customer_id  Recency  Frequency  Monetary
0       C0001       14          4    512.85
1       C0002       28          7   4644.90
2       C0003        7         22   3644.85
3       C0004        0          5    369.00
4       C0005      147          2    324.00
5       C0006       34          1     90.00
6       C0007        3         18   2600.00
7       C0008        3         15   3096.50
8       C0009       19         11   8424.80
9       C0010       66          4    450.00

RFM Statistics:
       Recency  Frequency  Monetary
count  1175.00    1175.00   1175.00
mean     33.55       9.46   2552.06
std      38.43       8.01   2367.31
min       0.00       1.00     12.00
25%       6.00       3.00    505.45
50%      19.00       5.00   2286.50
75%      46.00      17.00   3751.52
max     180.00      33.00  14766.55

REFERENCE VALIDATION
Total customers in RFM: 11

In [19]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("CLUSTER WITH K-MEANS")
print("="*80)

# Load RFM table
rfm_table = pd.read_csv('data/rfm_table.csv')

# ===== STEP 1: STANDARDIZE RFM =====
print("\nStep 1: Standardizing RFM data with StandardScaler...")

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_table[['Recency', 'Frequency', 'Monetary']])

print(f"✅ Standardized shape: {rfm_scaled.shape}")
print(f"Scaled data mean (should be ~0): {rfm_scaled.mean(axis=0).round(6)}")
print(f"Scaled data std (should be ~1): {rfm_scaled.std(axis=0).round(6)}")

# ===== STEP 2: FIT K-MEANS FOR K=2 TO K=8 =====
print("\n" + "="*80)
print("Step 2: Fitting K-Means for K=2 to K=8...")
print("="*80)

results = []

for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(rfm_scaled)
    
    inertia = kmeans.inertia_
    sil_score = silhouette_score(rfm_scaled, kmeans.labels_)
    
    results.append({
        'K': k,
        'Inertia': inertia,
        'Silhouette Score': sil_score
    })
    
    print(f"K={k}: Inertia={inertia:.1f}, Silhouette={sil_score:.4f}")

# ===== CREATE RESULTS TABLE =====
print("\n" + "="*80)
print("RESULTS TABLE")
print("="*80)

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# ===== COMPARE WITH REFERENCE =====
print("\n" + "="*80)
print("COMPARISON WITH REFERENCE (Tolerance ±5% inertia, ±0.05 silhouette)")
print("="*80)

reference = {
    2: {'inertia': 1820.8, 'silhouette': 0.465},
    3: {'inertia': 1268.2, 'silhouette': 0.445},
    4: {'inertia': 743.1, 'silhouette': 0.526},
    5: {'inertia': 617.4, 'silhouette': 0.481},
    6: {'inertia': 503.2, 'silhouette': 0.482},
    7: {'inertia': 408.5, 'silhouette': 0.416},
    8: {'inertia': 366.5, 'silhouette': 0.412}
}

print("\nK\tActual Inertia\tRef Inertia\tMatch\tActual Sil\tRef Sil\tMatch")
print("-"*80)

for k in range(2, 9):
    actual_row = results_df[results_df['K'] == k].iloc[0]
    actual_inertia = actual_row['Inertia']
    actual_sil = actual_row['Silhouette Score']
    
    ref_inertia = reference[k]['inertia']
    ref_sil = reference[k]['silhouette']
    
    # Check tolerance (±5% inertia, ±0.05 silhouette)
    inertia_match = abs(actual_inertia - ref_inertia) / ref_inertia <= 0.05
    sil_match = abs(actual_sil - ref_sil) <= 0.05
    
    print(f"{k}\t{actual_inertia:.1f}\t\t{ref_inertia:.1f}\t\t{'✅' if inertia_match else '❌'}\t"
          f"{actual_sil:.4f}\t{ref_sil:.4f}\t{'✅' if sil_match else '❌'}")

# ===== ELBOW ANALYSIS =====
print("\n" + "="*80)
print("ELBOW METHOD ANALYSIS")
print("="*80)

print("\nInertia values (looking for the elbow point):")
for k in range(2, 9):
    inertia = results_df[results_df['K'] == k]['Inertia'].values[0]
    print(f"K={k}: {inertia:.1f}")

print("\nInertia decrease from K to K+1:")
for i in range(len(results_df) - 1):
    k = results_df.iloc[i]['K']
    decrease = results_df.iloc[i]['Inertia'] - results_df.iloc[i+1]['Inertia']
    pct_decrease = (decrease / results_df.iloc[i]['Inertia']) * 100
    print(f"K={int(k)} to K={int(k+1)}: {decrease:.1f} ({pct_decrease:.1f}% decrease)")

# ===== CHOOSE OPTIMAL K =====
print("\n" + "="*80)
print("CHOOSING OPTIMAL K FROM {3, 4, 5}")
print("="*80)

print("\nComparison of K=3, K=4, K=5:")
for k in [3, 4, 5]:
    row = results_df[results_df['K'] == k].iloc[0]
    print(f"\nK={k}:")
    print(f"  Inertia: {row['Inertia']:.1f}")
    print(f"  Silhouette: {row['Silhouette Score']:.4f}")

# Recommendation
print("\n" + "-"*80)
print("ANALYSIS:")
print("-"*80)
print("""
K=3: Good inertia drop from K=2, moderate silhouette score (0.445). 
     Simpler model with 3 interpretable segments.

K=4: Highest silhouette score (0.526), excellent separation. 
     Sharp inertia drop. Best statistical quality.

K=5: Further inertia reduction (617.4), slightly lower silhouette (0.481). 
     More granular but approaching diminishing returns.

RECOMMENDATION: K=4 is optimal.
- Highest silhouette score (0.526) indicates best cluster separation
- Clear elbow visible between K=3 and K=4
- 4 segments are actionable for business campaigns
- Sweet spot between model complexity and interpretability
""")

# ===== TRAIN FINAL K-MEANS WITH K=4 =====
print("\n" + "="*80)
print("TRAINING FINAL K-MEANS WITH K=4")
print("="*80)

final_k = 4
final_kmeans = KMeans(n_clusters=final_k, n_init=10, random_state=42)
rfm_table['Cluster'] = final_kmeans.fit_predict(rfm_scaled)

print(f"\n✅ K-Means trained with K={final_k}")
print(f"Final Silhouette Score: {silhouette_score(rfm_scaled, rfm_table['Cluster']):.4f}")
print(f"\nCluster distribution:")
print(rfm_table['Cluster'].value_counts().sort_index())

# Save clustered data
rfm_table.to_csv('data/rfm_clustered.csv', index=False)
print("\n✅ RFM table with clusters saved to data/rfm_clustered.csv")

# ===== WHY NOT K=8? =====
print("\n" + "="*80)
print("WHY NOT K=8 DESPITE LOWEST INERTIA?")
print("="*80)

explanation = """
Inertia decreases monotonically with K because more clusters allow each data point 
to be closer to its nearest centroid—this is mathematical, not a sign of better 
clustering. K=8 creates 8 small, fragmented segments that overfit the data and are 
too numerous for the business to design distinct, actionable retention campaigns around. 
The silhouette score of K=8 (0.412) is also much lower than K=4 (0.526), indicating 
poor cluster cohesion and separation at that level.
"""

print(explanation)

print("\n✅ Task 2: K-Means Clustering Complete!")

CLUSTER WITH K-MEANS

Step 1: Standardizing RFM data with StandardScaler...
✅ Standardized shape: (1175, 3)
Scaled data mean (should be ~0): [-0.  0. -0.]
Scaled data std (should be ~1): [1. 1. 1.]

Step 2: Fitting K-Means for K=2 to K=8...
K=2: Inertia=1820.6, Silhouette=0.4650
K=3: Inertia=1268.1, Silhouette=0.4452
K=4: Inertia=742.9, Silhouette=0.5256
K=5: Inertia=617.3, Silhouette=0.4805
K=6: Inertia=503.1, Silhouette=0.4825
K=7: Inertia=408.4, Silhouette=0.4164
K=8: Inertia=366.5, Silhouette=0.4123

RESULTS TABLE
 K     Inertia  Silhouette Score
 2 1820.648479          0.464996
 3 1268.066769          0.445250
 4  742.927238          0.525624
 5  617.347166          0.480512
 6  503.142606          0.482546
 7  408.449959          0.416360
 8  366.508114          0.412319

COMPARISON WITH REFERENCE (Tolerance ±5% inertia, ±0.05 silhouette)

K	Actual Inertia	Ref Inertia	Match	Actual Sil	Ref Sil	Match
--------------------------------------------------------------------------------
2

In [20]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os

print("="*80)
print("PART B TASK 3: PROFILE, NAME AND PROJECT SEGMENTS")
print("="*80)

# Load data
rfm_table = pd.read_csv('data/rfm_clustered.csv')
customers = pd.read_csv('data/clean/customers.csv')

# ===== PART (a): REPORT SEGMENT SIZE AND RFM STATISTICS =====
print("\n" + "="*80)
print("PART (a): Segment Size and RFM Statistics")
print("="*80)

segment_stats = rfm_table.groupby('Cluster').agg({
    'customer_id': 'count',
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean'
}).round(2)

segment_stats.columns = ['Size', 'Mean Recency', 'Mean Frequency', 'Mean Monetary']
print("\nSegment Statistics:")
print(segment_stats)

# Verify against reference (K=4)
print("\n" + "-"*80)
print("Verification against Reference (K=4):")
print("-"*80)
print(f"Cluster 0: Size {int(segment_stats.loc[0, 'Size'])} (Expected ~436)")
print(f"Cluster 1: Size {int(segment_stats.loc[1, 'Size'])} (Expected 138)")
print(f"          Mean Monetary INR {segment_stats.loc[1, 'Mean Monetary']:.2f} (Expected ~7,130)")
print(f"          Mean Frequency {segment_stats.loc[1, 'Mean Frequency']:.1f} (Expected ~6.8)")
print(f"Cluster 2: Size {int(segment_stats.loc[2, 'Size'])} (Expected ~419)")
print(f"          Mean Frequency {segment_stats.loc[2, 'Mean Frequency']:.1f} (Expected ~19.4)")
print(f"          Mean Recency {segment_stats.loc[2, 'Mean Recency']:.1f} days (Expected ~8.8)")
print(f"Cluster 3: Size {int(segment_stats.loc[3, 'Size'])} (Expected ~182)")

# ===== PART (b): NAME SEGMENTS AND IDENTIFY PREMIUM =====
print("\n" + "="*80)
print("PART (b): Segment Naming and Premium Identification")
print("="*80)

print("\nNote: Cluster numbers are only labels with no inherent ordering.\n")

# Find segment with highest mean monetary (Premium)
premium_cluster = segment_stats['Mean Monetary'].idxmax()
premium_monetary = segment_stats.loc[premium_cluster, 'Mean Monetary']

# Define segment names and actions
segment_definitions = {
    0: {
        'name': 'Loyal Advocates',
        'action': 'Reward with VIP benefits and exclusive early access to new products',
        'is_premium': (0 == premium_cluster)
    },
    1: {
        'name': 'Premium Customers',
        'action': 'Provide personalized support and premium service tier',
        'is_premium': (1 == premium_cluster)
    },
    2: {
        'name': 'Frequent Buyers',
        'action': 'Engage with frequent deals and loyalty rewards program',
        'is_premium': (2 == premium_cluster)
    },
    3: {
        'name': 'Growth Opportunity',
        'action': 'Re-engage with targeted offers and incentives to increase frequency',
        'is_premium': (3 == premium_cluster)
    }
}

print("Segment Names and Recommended Actions:")
print("-"*80)

for cluster_id in sorted(segment_stats.index):
    stats = segment_stats.loc[cluster_id]
    definition = segment_definitions[cluster_id]
    
    print(f"\nCluster {cluster_id}: {definition['name']}")
    print(f"  Size: {int(stats['Size'])} customers")
    print(f"  Mean Recency: {stats['Mean Recency']:.1f} days")
    print(f"  Mean Frequency: {stats['Mean Frequency']:.1f} orders")
    print(f"  Mean Monetary: INR {stats['Mean Monetary']:.2f}")
    print(f"  Recommended Action: {definition['action']}")
    
    if definition['is_premium']:
        print(f"  🌟 PREMIUM SEGMENT (Highest Mean Monetary)")

# Add segment names to RFM table
rfm_table['Segment_Name'] = rfm_table['Cluster'].map(lambda x: segment_definitions[x]['name'])
rfm_table['Is_Premium'] = rfm_table['Cluster'].map(lambda x: segment_definitions[x]['is_premium'])

print(f"\n✅ Premium Segment: Cluster {premium_cluster} ({segment_definitions[premium_cluster]['name']})")
print(f"   Mean Monetary Value: INR {premium_monetary:.2f}")

# ===== PART (c): PCA ANALYSIS =====
print("\n" + "="*80)
print("PART (c): PCA Analysis and Projection")
print("="*80)

# Re-standardize RFM for PCA (use fresh StandardScaler)
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_table[['Recency', 'Frequency', 'Monetary']])

# Apply PCA with 2 components
pca = PCA(n_components=2)
rfm_pca = pca.fit_transform(rfm_scaled)

# Report explained variance
variance_ratio = pca.explained_variance_ratio_
print(f"\nExplained Variance Ratio:")
print(f"  PC1: {variance_ratio[0]:.4f} (Expected ~0.634, tolerance ±0.02)")
print(f"  PC2: {variance_ratio[1]:.4f} (Expected ~0.218, tolerance ±0.02)")
print(f"  Total: {variance_ratio.sum():.4f} (Expected ~0.852, tolerance ±0.02)")

# Verify against reference
pc1_match = abs(variance_ratio[0] - 0.634) <= 0.02
pc2_match = abs(variance_ratio[1] - 0.218) <= 0.02
total_match = abs(variance_ratio.sum() - 0.852) <= 0.02

print(f"\nVerification:")
print(f"  PC1 within tolerance: {'✅' if pc1_match else '❌'}")
print(f"  PC2 within tolerance: {'✅' if pc2_match else '❌'}")
print(f"  Total within tolerance: {'✅' if total_match else '❌'}")

# Add PCA components to RFM table
rfm_table['PCA1'] = rfm_pca[:, 0]
rfm_table['PCA2'] = rfm_pca[:, 1]

print(f"\n✅ PCA analysis complete")
print(f"   Shape of PCA projection: {rfm_pca.shape}")

# ===== CREATE OUTPUT FILE =====
print("\n" + "="*80)
print("Creating Output File: outputs/customer_segments.csv")
print("="*80)

# Create output directory if it doesn't exist
os.makedirs('outputs', exist_ok=True)

# Create output dataframe with required columns
output_df = pd.DataFrame({
    'customer_id': rfm_table['customer_id'],
    'recency': rfm_table['Recency'],
    'frequency': rfm_table['Frequency'],
    'monetary': rfm_table['Monetary'],
    'segment_id': rfm_table['Cluster'],
    'segment_name': rfm_table['Segment_Name'],
    'is_premium': rfm_table['Is_Premium']
})

# Save to CSV
output_df.to_csv('outputs/customer_segments.csv', index=False)
print(f"\n✅ Output file saved: outputs/customer_segments.csv")
print(f"   Shape: {output_df.shape}")
print(f"   Columns: {list(output_df.columns)}")

# Display sample
print(f"\nFirst 10 rows of output:")
print(output_df.head(10))

# Summary statistics of output
print(f"\nOutput Summary:")
print(f"  Total customers: {len(output_df)}")
print(f"  Segments: {output_df['segment_name'].unique().tolist()}")
print(f"  Premium customers: {output_df['is_premium'].sum()}")

# Save additional files for reference
rfm_table.to_csv('data/rfm_with_segments.csv', index=False)
print(f"\n✅ Enhanced RFM table saved: data/rfm_with_segments.csv")

# Save segment summary
segment_summary = output_df.groupby('segment_id').agg({
    'customer_id': 'count',
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': 'mean'
}).round(2)
segment_summary.columns = ['Size', 'Avg_Recency', 'Avg_Frequency', 'Avg_Monetary']
segment_summary['Segment_Name'] = segment_summary.index.map(lambda x: segment_definitions[x]['name'])
segment_summary['Is_Premium'] = segment_summary.index.map(lambda x: segment_definitions[x]['is_premium'])

print(f"\nSegment Summary:")
print(segment_summary)

segment_summary.to_csv('data/segment_summary.csv')
print(f"\n✅ Segment summary saved: data/segment_summary.csv")

print("\n" + "="*80)
print("✅ Task 3: Profile, Name and Project Segments Complete!")
print("="*80)

PART B TASK 3: PROFILE, NAME AND PROJECT SEGMENTS

PART (a): Segment Size and RFM Statistics

Segment Statistics:
         Size  Mean Recency  Mean Frequency  Mean Monetary
Cluster                                                   
0         436         28.85            3.77        1057.62
1         138         24.09            6.77        7129.70
2         419          8.75           19.40        3410.02
3         182        109.04            2.20         685.99

--------------------------------------------------------------------------------
Verification against Reference (K=4):
--------------------------------------------------------------------------------
Cluster 0: Size 436 (Expected ~436)
Cluster 1: Size 138 (Expected 138)
          Mean Monetary INR 7129.70 (Expected ~7,130)
          Mean Frequency 6.8 (Expected ~6.8)
Cluster 2: Size 419 (Expected ~419)
          Mean Frequency 19.4 (Expected ~19.4)
          Mean Recency 8.8 days (Expected ~8.8)
Cluster 3: Size 182 (Expected 

In [21]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os

print("="*80)
print("PART B TASK 3: PROFILE, NAME AND PROJECT SEGMENTS")
print("="*80)

# Load data
rfm_table = pd.read_csv('data/rfm_clustered.csv')
customers = pd.read_csv('data/clean/customers.csv')

# ===== PART (a): REPORT SEGMENT SIZE AND RFM STATISTICS =====
print("\n" + "="*80)
print("PART (a): Segment Size and RFM Statistics")
print("="*80)

segment_stats = rfm_table.groupby('Cluster').agg({
    'customer_id': 'count',
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean'
}).round(2)

segment_stats.columns = ['Size', 'Mean Recency', 'Mean Frequency', 'Mean Monetary']
print("\nSegment Statistics:")
print(segment_stats)

# Verify against reference (K=4)
print("\n" + "-"*80)
print("Verification against Reference (K=4):")
print("-"*80)
print(f"Cluster 0: Size {int(segment_stats.loc[0, 'Size'])} (Expected ~436)")
print(f"Cluster 1: Size {int(segment_stats.loc[1, 'Size'])} (Expected 138)")
print(f"          Mean Monetary INR {segment_stats.loc[1, 'Mean Monetary']:.2f} (Expected ~7,130)")
print(f"          Mean Frequency {segment_stats.loc[1, 'Mean Frequency']:.1f} (Expected ~6.8)")
print(f"Cluster 2: Size {int(segment_stats.loc[2, 'Size'])} (Expected ~419)")
print(f"          Mean Frequency {segment_stats.loc[2, 'Mean Frequency']:.1f} (Expected ~19.4)")
print(f"          Mean Recency {segment_stats.loc[2, 'Mean Recency']:.1f} days (Expected ~8.8)")
print(f"Cluster 3: Size {int(segment_stats.loc[3, 'Size'])} (Expected ~182)")

# ===== PART (b): NAME SEGMENTS AND IDENTIFY PREMIUM =====
print("\n" + "="*80)
print("PART (b): Segment Naming and Premium Identification")
print("="*80)

print("\nNote: Cluster numbers are only labels with no inherent ordering.\n")

# Find segment with highest mean monetary (Premium)
premium_cluster = segment_stats['Mean Monetary'].idxmax()
premium_monetary = segment_stats.loc[premium_cluster, 'Mean Monetary']

# Define segment names and actions
segment_definitions = {
    0: {
        'name': 'Occasional Shoppers',
        'action': 'Send small-basket offers and reminders to lift order frequency',
        'is_premium': (0 == premium_cluster)
    },
    1: {
        'name': 'Premium Customers',
        'action': 'Assign dedicated support and a premium loyalty tier',
        'is_premium': (1 == premium_cluster)
    },
    2: {
        'name': 'Loyal Regulars',
        'action': 'Retain with loyalty rewards, bundles and early access to deals',
        'is_premium': (2 == premium_cluster)
    },
    3: {
        'name': 'Lapsing Customers',
        'action': 'Run a win-back campaign with time-limited incentives',
        'is_premium': (3 == premium_cluster)
    }
}

print("Segment Names and Recommended Actions:")
print("-"*80)

for cluster_id in sorted(segment_stats.index):
    stats = segment_stats.loc[cluster_id]
    definition = segment_definitions[cluster_id]
    
    print(f"\nCluster {cluster_id}: {definition['name']}")
    print(f"  Size: {int(stats['Size'])} customers")
    print(f"  Mean Recency: {stats['Mean Recency']:.1f} days")
    print(f"  Mean Frequency: {stats['Mean Frequency']:.1f} orders")
    print(f"  Mean Monetary: INR {stats['Mean Monetary']:.2f}")
    print(f"  Recommended Action: {definition['action']}")
    
    if definition['is_premium']:
        print(f"  🌟 PREMIUM SEGMENT (Highest Mean Monetary)")

# Add segment names to RFM table
rfm_table['Segment_Name'] = rfm_table['Cluster'].map(lambda x: segment_definitions[x]['name'])
rfm_table['Is_Premium'] = rfm_table['Cluster'].map(lambda x: segment_definitions[x]['is_premium'])

print(f"\n✅ Premium Segment: Cluster {premium_cluster} ({segment_definitions[premium_cluster]['name']})")
print(f"   Mean Monetary Value: INR {premium_monetary:.2f}")

# ===== PART (c): PCA ANALYSIS =====
print("\n" + "="*80)
print("PART (c): PCA Analysis and Projection")
print("="*80)

# Re-standardize RFM for PCA (use fresh StandardScaler)
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_table[['Recency', 'Frequency', 'Monetary']])

# Apply PCA with 2 components
pca = PCA(n_components=2)
rfm_pca = pca.fit_transform(rfm_scaled)

# Report explained variance
variance_ratio = pca.explained_variance_ratio_
print(f"\nExplained Variance Ratio:")
print(f"  PC1: {variance_ratio[0]:.4f} (Expected ~0.634, tolerance ±0.02)")
print(f"  PC2: {variance_ratio[1]:.4f} (Expected ~0.218, tolerance ±0.02)")
print(f"  Total: {variance_ratio.sum():.4f} (Expected ~0.852, tolerance ±0.02)")

# Verify against reference
pc1_match = abs(variance_ratio[0] - 0.634) <= 0.02
pc2_match = abs(variance_ratio[1] - 0.218) <= 0.02
total_match = abs(variance_ratio.sum() - 0.852) <= 0.02

print(f"\nVerification:")
print(f"  PC1 within tolerance: {'✅' if pc1_match else '❌'}")
print(f"  PC2 within tolerance: {'✅' if pc2_match else '❌'}")
print(f"  Total within tolerance: {'✅' if total_match else '❌'}")

# Add PCA components to RFM table
rfm_table['PCA1'] = rfm_pca[:, 0]
rfm_table['PCA2'] = rfm_pca[:, 1]

print(f"\n✅ PCA analysis complete")
print(f"   Shape of PCA projection: {rfm_pca.shape}")

# ===== CREATE OUTPUT FILE =====
print("\n" + "="*80)
print("Creating Output File: outputs/customer_segments.csv")
print("="*80)

# Create output directory if it doesn't exist
os.makedirs('outputs', exist_ok=True)

# Create output dataframe with required columns
output_df = pd.DataFrame({
    'customer_id': rfm_table['customer_id'],
    'recency': rfm_table['Recency'],
    'frequency': rfm_table['Frequency'],
    'monetary': rfm_table['Monetary'],
    'segment_id': rfm_table['Cluster'],
    'segment_name': rfm_table['Segment_Name'],
    'is_premium': rfm_table['Is_Premium']
})

# Save to CSV
output_df.to_csv('outputs/customer_segments.csv', index=False)
print(f"\n✅ Output file saved: outputs/customer_segments.csv")
print(f"   Shape: {output_df.shape}")
print(f"   Columns: {list(output_df.columns)}")

# Display sample
print(f"\nFirst 10 rows of output:")
print(output_df.head(10))

# Summary statistics of output
print(f"\nOutput Summary:")
print(f"  Total customers: {len(output_df)}")
print(f"  Segments: {output_df['segment_name'].unique().tolist()}")
print(f"  Premium customers: {output_df['is_premium'].sum()}")

# Save additional files for reference
rfm_table.to_csv('data/rfm_with_segments.csv', index=False)
print(f"\n✅ Enhanced RFM table saved: data/rfm_with_segments.csv")

# Save segment summary
segment_summary = output_df.groupby('segment_id').agg({
    'customer_id': 'count',
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': 'mean'
}).round(2)
segment_summary.columns = ['Size', 'Avg_Recency', 'Avg_Frequency', 'Avg_Monetary']
segment_summary['Segment_Name'] = segment_summary.index.map(lambda x: segment_definitions[x]['name'])
segment_summary['Is_Premium'] = segment_summary.index.map(lambda x: segment_definitions[x]['is_premium'])

print(f"\nSegment Summary:")
print(segment_summary)

segment_summary.to_csv('data/segment_summary.csv')
print(f"\n✅ Segment summary saved: data/segment_summary.csv")


PART B TASK 3: PROFILE, NAME AND PROJECT SEGMENTS

PART (a): Segment Size and RFM Statistics

Segment Statistics:
         Size  Mean Recency  Mean Frequency  Mean Monetary
Cluster                                                   
0         436         28.85            3.77        1057.62
1         138         24.09            6.77        7129.70
2         419          8.75           19.40        3410.02
3         182        109.04            2.20         685.99

--------------------------------------------------------------------------------
Verification against Reference (K=4):
--------------------------------------------------------------------------------
Cluster 0: Size 436 (Expected ~436)
Cluster 1: Size 138 (Expected 138)
          Mean Monetary INR 7129.70 (Expected ~7,130)
          Mean Frequency 6.8 (Expected ~6.8)
Cluster 2: Size 419 (Expected ~419)
          Mean Frequency 19.4 (Expected ~19.4)
          Mean Recency 8.8 days (Expected ~8.8)
Cluster 3: Size 182 (Expected 

In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

CUTOFF = pd.Timestamp('2025-09-01')
LABEL_END = pd.Timestamp('2025-10-01')   # label window: 2025-09-01 to 2025-09-30

# ===== LOAD CLEAN DATA =====
orders = pd.read_csv('data/clean/orders.csv')
orders['order_ts'] = pd.to_datetime(orders['order_ts'])
customers = pd.read_csv('data/clean/customers.csv')
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
products = pd.read_csv('data/clean/products.csv')

# orders.csv already carries category/price_inr from Part A; refresh them from products to avoid _x/_y clashes
orders = orders.drop(columns=['category', 'price_inr'], errors='ignore')
orders = orders.merge(products[['product_id', 'category']], on='product_id', how='left')

print("="*80)
print("PART B TASK 4: DEFINE THE LAPSE PROBLEM")
print("="*80)

# ===== STEP 1: HISTORY = ORDERS BEFORE CUTOFF ONLY (no leakage) =====
hist = orders[orders['order_ts'] < CUTOFF]
completed_hist = hist[hist['status'] == 'completed']
final_hist = hist[hist['status'].isin(['completed', 'cancelled'])]

# ===== STEP 2: COHORT = at least one completed order before cutoff =====
cohort_ids = pd.Index(completed_hist['customer_id'].unique(), name='customer_id')
print(f"\nCohort size: {len(cohort_ids)} (Expected: 1156)")

# ===== STEP 3: LABEL = 1 if NO completed order in Sept 2025 =====
sept_active = orders[
    (orders['status'] == 'completed') &
    (orders['order_ts'] >= CUTOFF) &
    (orders['order_ts'] < LABEL_END)
]['customer_id'].unique()

label = pd.DataFrame({'customer_id': cohort_ids})
label['lapsed'] = (~label['customer_id'].isin(sept_active)).astype(int)
n_lapsed = label['lapsed'].sum()
print(f"Lapsed: {n_lapsed} ({n_lapsed/len(label)*100:.2f}%) (Expected: 443, 38.32%)")

# ===== STEP 4: FEATURES FROM PRE-CUTOFF HISTORY =====
g = completed_hist.groupby('customer_id')
feat = g.agg(
    last_ts=('order_ts', 'max'),
    frequency=('order_id', 'count'),
    monetary=('order_amount', 'sum'),
    avg_discount_pct=('discount_pct', 'mean'),
    avg_delivery_minutes=('delivery_minutes', 'mean'),
).reset_index()
feat['recency'] = (CUTOFF - feat['last_ts']).dt.days
feat['monetary'] = feat['monetary'].round(2)

# Most frequent category among completed pre-cutoff orders
top_cat = (g['category']
           .agg(lambda s: s.mode().iloc[0])
           .rename('top_category')
           .reset_index())

# Cancellation rate = cancelled / (completed + cancelled), before cutoff
cancel = (final_hist.groupby('customer_id')['status']
          .agg(cancellation_rate=lambda s: (s == 'cancelled').mean())
          .reset_index())

# ===== STEP 5: CUSTOMER ATTRIBUTES =====
cust = customers[['customer_id', 'city', 'gender', 'age', 'signup_date']].copy()
cust['tenure_days'] = (CUTOFF - cust['signup_date']).dt.days

# ===== STEP 6: ASSEMBLE DATASET =====
df = (label
      .merge(feat.drop(columns=['last_ts']), on='customer_id', how='left')
      .merge(top_cat, on='customer_id', how='left')
      .merge(cancel, on='customer_id', how='left')
      .merge(cust, on='customer_id', how='left'))

df = df.drop(columns=['signup_date'])
df.to_csv('data/lapse_dataset.csv', index=False)

print(f"\nDataset shape: {df.shape}")
print("Missing values per column:")
print(df.isna().sum()[lambda s: s > 0])

PART B TASK 4: DEFINE THE LAPSE PROBLEM

Cohort size: 1156 (Expected: 1156)
Lapsed: 443 (38.32%) (Expected: 443, 38.32%)

Dataset shape: (1156, 13)
Missing values per column:
Series([], dtype: int64)


In [23]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Features: everything except the ID and the label
# (segment labels from Task 3 are NOT included)
feature_cols = ['recency', 'frequency', 'monetary', 'avg_discount_pct',
                'cancellation_rate', 'avg_delivery_minutes', 'top_category',
                'city', 'age', 'gender', 'tenure_days']

X = df[feature_cols]
y = df['lapsed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("="*80)
print("TRAIN / TEST SPLIT")
print("="*80)
print(f"Training rows: {len(X_train)} (Expected: 924)")
print(f"Test rows: {len(X_test)} (Expected: 232)")
print(f"Lapsed share in train: {y_train.mean():.4f}")
print(f"Lapsed share in test:  {y_test.mean():.4f}")

# ===== BASELINE: ALWAYS PREDICT "NOT LAPSED" =====
baseline_pred = [0] * len(y_test)
baseline_acc = accuracy_score(y_test, baseline_pred)
baseline_f1 = f1_score(y_test, baseline_pred, pos_label=1, zero_division=0)

print("\n" + "="*80)
print("BASELINE: ALWAYS PREDICT NOT LAPSED")
print("="*80)
print(f"Accuracy: {baseline_acc*100:.1f}% (Expected: 61.6%)")
print(f"Lapsed-class F1: {baseline_f1:.2f} (Expected: 0)")


TRAIN / TEST SPLIT
Training rows: 924 (Expected: 924)
Test rows: 232 (Expected: 232)
Lapsed share in train: 0.3831
Lapsed share in test:  0.3836

BASELINE: ALWAYS PREDICT NOT LAPSED
Accuracy: 61.6% (Expected: 61.6%)
Lapsed-class F1: 0.00 (Expected: 0)


## Part B Task 5: Preprocessing Without Leakage.
The learned frequency mean (8.12) comes from the 924 training customers the scaler was fitted on, while the full-cohort mean (8.03) includes the 232 test customers the scaler never saw."

In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
df = pd.read_csv('data/lapse_dataset.csv')

# ===== COLUMN GROUPS =====
nominal_cols = ['city', 'gender', 'top_category']          # one-hot encode, k-1 columns
numeric_cols = ['recency', 'frequency', 'monetary', 'avg_discount_pct',
                'cancellation_rate', 'avg_delivery_minutes', 'age', 'tenure_days']
feature_cols = numeric_cols + nominal_cols

X = df[feature_cols]
y = df['lapsed']

# Same split as Task 4
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ===== PREPROCESSOR (fitted on training rows only, via Pipeline) =====
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), nominal_cols),
    ]
)

preprocess_pipe = Pipeline([('preprocess', preprocessor)])

# fit ONLY on training data, then apply to both
X_train_prep = preprocess_pipe.fit_transform(X_train)
X_test_prep = preprocess_pipe.transform(X_test)

print("="*80)
print("PART B TASK 5: PREPROCESSING WITHOUT LEAKAGE")
print("="*80)
print(f"Train matrix shape: {X_train_prep.shape}")
print(f"Test matrix shape:  {X_test_prep.shape}")

feature_names = preprocess_pipe.named_steps['preprocess'].get_feature_names_out()
print(f"\nTransformed feature names ({len(feature_names)}):")
for name in feature_names:
    print(f"  - {name}")

# ===== LEARNED SCALER MEAN vs FULL-COHORT MEAN OF FREQUENCY =====
scaler = preprocess_pipe.named_steps['preprocess'].named_transformers_['num']
freq_idx = numeric_cols.index('frequency')
learned_freq_mean = scaler.mean_[freq_idx]
full_freq_mean = df['frequency'].mean()

print("\n" + "-"*80)
print("FREQUENCY MEAN CHECK")
print("-"*80)
print(f"Learned mean of frequency (scaler, training rows only): {learned_freq_mean:.4f}")
print(f"Full-cohort mean of frequency (all 1156 customers):     {full_freq_mean:.4f}")

# ===== NO MISSING VALUES IN FEATURE COLUMNS =====
assert X_train[feature_cols].isna().sum().sum() == 0, "Missing values in train features"
assert X_test[feature_cols].isna().sum().sum() == 0, "Missing values in test features"
assert not np.isnan(X_train_prep).any(), "NaN after preprocessing (train)"
assert not np.isnan(X_test_prep).any(), "NaN after preprocessing (test)"
print("\n✅ No missing values in any feature column (raw or after preprocessing)")

PART B TASK 5: PREPROCESSING WITHOUT LEAKAGE
Train matrix shape: (924, 19)
Test matrix shape:  (232, 19)

Transformed feature names (19):
  - num__recency
  - num__frequency
  - num__monetary
  - num__avg_discount_pct
  - num__cancellation_rate
  - num__avg_delivery_minutes
  - num__age
  - num__tenure_days
  - cat__city_Chennai
  - cat__city_Delhi
  - cat__city_Hyderabad
  - cat__city_Mumbai
  - cat__city_Pune
  - cat__gender_male
  - cat__gender_unknown
  - cat__top_category_Electronics
  - cat__top_category_Grocery
  - cat__top_category_Personal Care
  - cat__top_category_Snacks

--------------------------------------------------------------------------------
FREQUENCY MEAN CHECK
--------------------------------------------------------------------------------
Learned mean of frequency (scaler, training rows only): 8.1158
Full-cohort mean of frequency (all 1156 customers):     8.0285

✅ No missing values in any feature column (raw or after preprocessing)


In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, precision_score,
                             recall_score, f1_score, accuracy_score)
import warnings
from scipy.optimize import OptimizeWarning
warnings.filterwarnings('ignore', category=OptimizeWarning)

# ===== LOAD AND SPLIT (same as Tasks 4-5) =====
df = pd.read_csv('data/lapse_dataset.csv')

numeric_cols = ['recency', 'frequency', 'monetary', 'avg_discount_pct',
                'cancellation_rate', 'avg_delivery_minutes', 'age', 'tenure_days']
nominal_cols = ['city', 'gender', 'top_category']
feature_cols = numeric_cols + nominal_cols

X = df[feature_cols]
y = df['lapsed']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

def make_preprocessor():
    # Fresh, unfitted preprocessor so each pipeline fits it on training folds only
    return ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), nominal_cols),
    ])

print("="*80)
print("PART B TASK 6: TRAIN, TUNE AND EVALUATE")
print("="*80)

# ===== (a) BASELINE =====
print("\n(a) BASELINE: always predict 'not lapsed'")
baseline = DummyClassifier(strategy='constant', constant=0).fit(X_train, y_train)
b_pred = baseline.predict(X_test)
print(f"  Accuracy: {accuracy_score(y_test, b_pred):.3f}")
print(f"  Lapsed F1: {f1_score(y_test, b_pred, zero_division=0):.3f}")

# ===== (b) LOGISTIC REGRESSION GRID SEARCH =====
print("\n(b) LOGISTIC REGRESSION: GridSearchCV over C")
lr_pipe = Pipeline([
    ('preprocess', make_preprocessor()),
    ('clf', LogisticRegression(max_iter=1000)),
])
lr_grid = GridSearchCV(
    lr_pipe,
    param_grid={'clf__C': [0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='f1'
)
lr_grid.fit(X_train, y_train)     # training data only
print(f"  Best params: {lr_grid.best_params_}")
print(f"  Best CV F1: {lr_grid.best_score_:.3f}")

# ===== (c) RANDOM FOREST GRID SEARCH =====
print("\n(c) RANDOM FOREST: GridSearchCV over max_depth and min_samples_leaf")
rf_pipe = Pipeline([
    ('preprocess', make_preprocessor()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42)),
])
rf_grid = GridSearchCV(
    rf_pipe,
    param_grid={'clf__max_depth': [3, 5, 8],
                'clf__min_samples_leaf': [5, 10, 20]},
    cv=5, scoring='f1'
)
rf_grid.fit(X_train, y_train)     # training data only (45 fits)
print(f"  Best params: {rf_grid.best_params_}")
print(f"  Best CV F1: {rf_grid.best_score_:.3f}")

# ===== (d) UNTOUCHED TEST SET EVALUATION =====
print("\n(d) TEST SET EVALUATION (run only after tuning)")
results = {}
for name, model in [('Logistic Regression', lr_grid.best_estimator_),
                    ('Random Forest', rf_grid.best_estimator_)]:
    pred = model.predict(X_test)
    cm = confusion_matrix(y_test, pred)
    results[name] = {
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
    }
    print(f"\n{name}")
    print("  Confusion matrix (rows = actual, cols = predicted):")
    print("                 Pred 0   Pred 1")
    print(f"  Actual 0       {cm[0,0]:>5}    {cm[0,1]:>5}")
    print(f"  Actual 1       {cm[1,0]:>5}    {cm[1,1]:>5}")
    print(f"  Lapsed precision: {results[name]['precision']:.3f}")
    print(f"  Lapsed recall:    {results[name]['recall']:.3f}")
    print(f"  Lapsed F1:        {results[name]['f1']:.3f}")

# ===== SELECTION CHECK =====
print("\n" + "-"*80)
best_name = max(results, key=lambda k: results[k]['f1'])
print(f"Higher test lapsed F1: {best_name} ({results[best_name]['f1']:.3f})")
print("Meets minimum test F1 of 0.55:",
      all(r['f1'] >= 0.55 for r in results.values()))

PART B TASK 6: TRAIN, TUNE AND EVALUATE

(a) BASELINE: always predict 'not lapsed'
  Accuracy: 0.616
  Lapsed F1: 0.000

(b) LOGISTIC REGRESSION: GridSearchCV over C
  Best params: {'clf__C': 0.1}
  Best CV F1: 0.639

(c) RANDOM FOREST: GridSearchCV over max_depth and min_samples_leaf
  Best params: {'clf__max_depth': 3, 'clf__min_samples_leaf': 5}
  Best CV F1: 0.675

(d) TEST SET EVALUATION (run only after tuning)

Logistic Regression
  Confusion matrix (rows = actual, cols = predicted):
                 Pred 0   Pred 1
  Actual 0         105       38
  Actual 1          17       72
  Lapsed precision: 0.655
  Lapsed recall:    0.809
  Lapsed F1:        0.724

Random Forest
  Confusion matrix (rows = actual, cols = predicted):
                 Pred 0   Pred 1
  Actual 0         112       31
  Actual 1          23       66
  Lapsed precision: 0.680
  Lapsed recall:    0.742
  Lapsed F1:        0.710

--------------------------------------------------------------------------------
High

##PART B - Train, tune and evaluate 
Missing a lapsing customer costs more than a wrongly targeted loyal one, because a missed customer may churn without a win-back offer, while a wrong target only costs the price of an unnecessary incentive. Logistic regression gives the higher lapsed recall (0.809 vs 0.742), so it misses fewer lapsing customers, which is the cost that matters most here. Its precision (0.655) is also close to the forest's (0.680), so the extra false alarms are a modest price. The two models' test F1 scores are close (0.724 vs 0.710) and the random forest has higher CV F1, but the difference is within noise on 89 lapsed cases. We choose logistic regression because it has comparable performance, higher recall, and coefficients the retention team can read

In [26]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.utils import resample

print("="*80)
print("PART B TASK 7: CLASS-BALANCE EXPERIMENT")
print("="*80)

# ===== STEP 1: OVER-SAMPLE THE TRAINING SET ONLY =====
train_df = X_train.copy()
train_df['lapsed'] = y_train.values

majority = train_df[train_df['lapsed'] == 0]
minority = train_df[train_df['lapsed'] == 1]

# Add random duplicates of lapsed rows (with replacement) until counts are equal.
# Original rows are kept; only the extra copies are added.
n_extra = len(majority) - len(minority)
minority_extra = resample(minority, replace=True, n_samples=n_extra, random_state=42)

balanced_train = pd.concat([train_df, minority_extra]).sample(frac=1, random_state=42)

X_bal = balanced_train[feature_cols]
y_bal = balanced_train['lapsed']

print(f"\nOriginal training counts: {y_train.value_counts().sort_index().to_dict()}")
print(f"Balanced training counts: {y_bal.value_counts().sort_index().to_dict()} (Expected: 570 each)")
print(f"Original training rows: {len(X_train)}, balanced training rows: {len(X_bal)}")

# ===== STEP 2: REFIT BEST LOGISTIC REGRESSION ON BALANCED DATA =====
best_lr_unbalanced = lr_grid.best_estimator_
best_C = lr_grid.best_params_['clf__C']
print(f"\nBest C from Task 6: {best_C}")

balanced_lr = clone(best_lr_unbalanced)   # same preprocessing + same C, unfitted copy
balanced_lr.fit(X_bal, y_bal)             # preprocessing refits on balanced training rows only

# ===== STEP 3: COMPARE ON THE ORIGINAL, UNCHANGED TEST SET =====
def lapsed_metrics(model, X, y):
    pred = model.predict(X)
    return {
        'precision': precision_score(y, pred, zero_division=0),
        'recall': recall_score(y, pred, zero_division=0),
        'f1': f1_score(y, pred, zero_division=0),
        'cm': confusion_matrix(y, pred),
    }

m_orig = lapsed_metrics(best_lr_unbalanced, X_test, y_test)
m_bal = lapsed_metrics(balanced_lr, X_test, y_test)

print("\n" + "-"*80)
print("TEST SET COMPARISON (original, unchanged test set)")
print("-"*80)
print(f"{'Metric':<12}{'Unbalanced':>12}{'Balanced':>12}{'Change':>12}")
for metric in ['precision', 'recall', 'f1']:
    a, b = m_orig[metric], m_bal[metric]
    print(f"{metric:<12}{a:>12.3f}{b:>12.3f}{b - a:>+12.3f}")

print("\nConfusion matrix, unbalanced (rows = actual, cols = predicted):")
print(m_orig['cm'])
print("\nConfusion matrix, balanced (rows = actual, cols = predicted):")
print(m_bal['cm'])

# Which metric moved most (absolute change)
changes = {m: abs(m_bal[m] - m_orig[m]) for m in ['precision', 'recall', 'f1']}
print(f"\nMetric that moved most: {max(changes, key=changes.get)}")

PART B TASK 7: CLASS-BALANCE EXPERIMENT

Original training counts: {0: 570, 1: 354}
Balanced training counts: {0: 570, 1: 570} (Expected: 570 each)
Original training rows: 924, balanced training rows: 1140

Best C from Task 6: 0.1

--------------------------------------------------------------------------------
TEST SET COMPARISON (original, unchanged test set)
--------------------------------------------------------------------------------
Metric        Unbalanced    Balanced      Change
precision          0.655       0.587      -0.068
recall             0.809       0.910      +0.101
f1                 0.724       0.714      -0.010

Confusion matrix, unbalanced (rows = actual, cols = predicted):
[[105  38]
 [ 17  72]]

Confusion matrix, balanced (rows = actual, cols = predicted):
[[86 57]
 [ 8 81]]

Metric that moved most: recall


## PART B - Class-balance experiment
Oversampling the training set raised lapsed-class recall from 0.809 to 0.910, the largest change, while precision fell from 0.655 to 0.587 and F1 stayed about the same (0.724 vs 0.714). For the retention team, the balanced model catches 9 more lapsing customers but wrongly targets 19 more loyal ones. Because F1 did not improve, we keep the original logistic regression as the main model and report the balanced version as a sensitivity check. If the campaign budget can absorb extra offers, the balanced model is the better choice for recall.

In [27]:
import os
import joblib
import numpy as np
import pandas as pd

os.makedirs('outputs', exist_ok=True)

print("="*80)
print("PART B TASK 8: INTERPRET, PERSIST AND USE")
print("="*80)

# ===== (a) INTERPRET =====
rf_model = rf_grid.best_estimator_
lr_model = lr_grid.best_estimator_

# Feature names after preprocessing (same for both models)
feat_names = lr_model.named_steps['preprocess'].get_feature_names_out()

# Random forest: top 5 features by importance (unsigned)
rf_imp = pd.Series(rf_model.named_steps['clf'].feature_importances_, index=feat_names)
print("\n(a) Top 5 random forest features (feature_importances_):")
print(rf_imp.sort_values(ascending=False).head(5).round(4).to_string())

# Logistic regression: signed coefficients (positive = raises lapse odds)
lr_coef = pd.Series(lr_model.named_steps['clf'].coef_[0], index=feat_names)
print("\nLogistic regression coefficients (sorted by absolute size):")
print(lr_coef.reindex(lr_coef.abs().sort_values(ascending=False).index).round(4).to_string())

# ===== (b) PERSIST AND RELOAD =====
# Selected model = logistic regression (chosen in Task 6)
joblib.dump(lr_model, 'outputs/lapse_model.joblib')
reloaded = joblib.load('outputs/lapse_model.joblib')

pred_orig = lr_model.predict(X_test)
pred_reload = reloaded.predict(X_test)
prob_orig = lr_model.predict_proba(X_test)[:, 1]
prob_reload = reloaded.predict_proba(X_test)[:, 1]

print("\n(b) Reload check:")
print(f"  Predictions identical: {np.array_equal(pred_orig, pred_reload)}")
print(f"  Probabilities identical: {np.allclose(prob_orig, prob_reload, rtol=0, atol=0)}")
assert np.array_equal(pred_orig, pred_reload), "Reloaded predictions differ"
assert np.array_equal(prob_orig, prob_reload), "Reloaded probabilities differ"

# ===== (c) RETENTION LIST FOR PREMIUM CUSTOMERS =====
# Premium flag is a post-hoc whole-window label (includes the September window).
# The retention list is illustrative, not a leakage-free evaluation.
seg = pd.read_csv('outputs/customer_segments.csv')
premium_ids = set(seg.loc[seg['is_premium'] == True, 'customer_id'])

test_ids = df.loc[X_test.index, 'customer_id'].values
test_df = pd.DataFrame({
    'customer_id': test_ids,
    'lapse_probability': prob_orig,
    'actually_lapsed': y_test.values,
})

premium_test = test_df[test_df['customer_id'].isin(premium_ids)].copy()
premium_test = premium_test.sort_values('lapse_probability', ascending=False)
top10 = premium_test.head(10).reset_index(drop=True)
top10['lapse_probability'] = top10['lapse_probability'].round(4)

top10.to_csv('outputs/retention_list.csv', index=False)

print(f"\n(c) Premium customers in test set: {len(premium_test)}")
print(f"Rows written to outputs/retention_list.csv: {len(top10)}")
print(f"Of these, truly lapsed (actually_lapsed = 1): {int(top10['actually_lapsed'].sum())} of {len(top10)}")
print(top10.to_string(index=False))

print("\n" + "-"*80)
print("NOTE: The Premium flag comes from whole-window RFM, which includes the")
print("September label window. This list is illustrative, not a leakage-free evaluation.")
print("-"*80)


PART B TASK 8: INTERPRET, PERSIST AND USE

(a) Top 5 random forest features (feature_importances_):
num__frequency            0.3522
num__monetary             0.2887
num__recency              0.1150
num__cancellation_rate    0.0531
num__avg_discount_pct     0.0407

Logistic regression coefficients (sorted by absolute size):
num__frequency                    -1.2811
cat__top_category_Electronics     -0.3782
cat__top_category_Snacks          -0.2026
cat__top_category_Grocery         -0.1997
cat__city_Delhi                   -0.1838
cat__city_Pune                     0.1128
cat__gender_male                  -0.1009
num__cancellation_rate            -0.1007
cat__city_Hyderabad               -0.0942
cat__gender_unknown               -0.0781
cat__top_category_Personal Care    0.0671
num__age                           0.0476
num__tenure_days                  -0.0460
num__monetary                      0.0294
num__recency                      -0.0285
num__avg_discount_pct              0.0266
nu

**##Part C — Support Ticket Triage with Guardrails**

In [62]:
import pandas as pd

# Load the support tickets file generated by generate_data.py
tickets = pd.read_csv("data/raw/tickets.csv")

# Show the first 5 rows
print(tickets.head())

# Show the column names
print(tickets.columns)

  ticket_id customer_id    order_id  \
0     T0024       C0486  ORD-105035   
1     T0073       C0080  ORD-100816   
2     T0095       C1087  ORD-111156   
3     T0159       C1172         NaN   
4     T0080       C0910  ORD-109516   

                                                text true_category  
0  There is an unauthorized transaction on my car...         fraud  
1  The invoice for ORD-100816 shows a wrong amoun...       billing  
2  The payment page shows an error and the app fr...     technical  
3                                                NaN          none  
4  I was charged for order ORD-109516 that I neve...         fraud  
Index(['ticket_id', 'customer_id', 'order_id', 'text', 'true_category'],
      dtype='str')


In [63]:
# 1. How many tickets fall into each category?
print(tickets["true_category"].value_counts())

# 2. How many tickets have missing text or missing category?
print(tickets[["text", "true_category"]].isna().sum())

# 3. How many rows have the 'none' label?
print("Rows labeled 'none':", (tickets["true_category"] == "none").sum())

true_category
delivery     43
fraud        39
billing      39
technical    39
none          4
Name: count, dtype: int64
text             4
true_category    0
dtype: int64
Rows labeled 'none': 4


In [64]:
import pandas as pd
from mock_llm import est_tokens  # token estimator supplied in mock_llm.py

MAX_INPUT_TOKENS = 400

# Requirement path (the earlier data/tickets.csv path may not match your folder)
tickets = pd.read_csv("data/raw/tickets.csv")

def input_guard(text):
    """Return a skip/flag status if the ticket must NOT go to the model, else None."""
    # (a) empty or whitespace-only text (NaN counts as empty)
    if pd.isna(text) or str(text).strip() == "":
        return "skipped_empty"
    # (b) too long for the token budget
    if est_tokens(str(text)) > MAX_INPUT_TOKENS:
        return "flagged_long"
    return None  # safe to send to the model

# Process in file order; iterrows() keeps the original order
guard_results = []
for _, row in tickets.iterrows():
    status = input_guard(row["text"])
    guard_results.append({
        "ticket_id": row["ticket_id"],
        "guard_status": status or "passed",
    })

guard_df = pd.DataFrame(guard_results)

# Verify against the reference numbers
print(guard_df["guard_status"].value_counts())
print("Total rows:", len(guard_df))  # should still be 164

guard_status
passed           156
skipped_empty      4
flagged_long       4
Name: count, dtype: int64
Total rows: 164


In [65]:
import re
import pandas as pd

# Patterns exactly as specified, applied in this order
PATTERNS = [
    ("EMAIL",    re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"), "[EMAIL]"),
    ("PHONE",    re.compile(r"\b[6-9]\d{9}\b"),          "[PHONE]"),
    ("ORDER_ID", re.compile(r"ORD-\d{6}"),               "[ORDER_ID]"),
]

def redact(text):
    """Return (redacted_text, substitution_counts) for one ticket."""
    if pd.isna(text):          # empty ticket -> empty string
        text = ""
    text = str(text)
    counts = {}
    for name, pattern, placeholder in PATTERNS:
        text, n = pattern.subn(placeholder, text)   # subn returns (new_text, count)
        counts[name] = n
    return text, counts

# Redact all 164 tickets in file order
redacted_texts = []
totals = {"EMAIL": 0, "PHONE": 0, "ORDER_ID": 0}

for text in tickets["text"]:
    clean, counts = redact(text)
    redacted_texts.append(clean)
    for name in totals:
        totals[name] += counts[name]

tickets["text_redacted"] = redacted_texts

# Report substitutions per type and in total
print("Substitutions per type:", totals)
print("Total substitutions:", sum(totals.values()))

# Verification: re-applying the patterns to redacted text should find nothing
leftover = {
    name: sum(len(pattern.findall(t)) for t in tickets["text_redacted"])
    for name, pattern, _ in PATTERNS
}
print("Matches left after redaction:", leftover)
print("Total left:", sum(leftover.values()))

Substitutions per type: {'EMAIL': 55, 'PHONE': 91, 'ORDER_ID': 132}
Total substitutions: 278
Matches left after redaction: {'EMAIL': 0, 'PHONE': 0, 'ORDER_ID': 0}
Total left: 0


In [66]:
# Zero-shot: instructions only, no examples
ZERO_SHOT_PROMPT = """You are a support ticket triage assistant for Swiggy Instamart.
CLASSIFY the customer's ticket into exactly one of these categories:
- billing
- delivery
- technical
- fraud

Reply with JSON only, no extra text, in this form:
{"category": "...", "reason": "..."}
"""

# Few-shot: same instructions, plus labelled examples
# The examples use placeholders like [ORDER_ID] and contain no real customer data.
FEW_SHOT_PROMPT = """You are a support ticket triage assistant for Swiggy Instamart.
CLASSIFY the customer's ticket into exactly one of these categories:
- billing
- delivery
- technical
- fraud

Reply with JSON only, no extra text, in this form:
{"category": "...", "reason": "..."}

EXAMPLES:
Ticket: "I was charged twice for order [ORDER_ID] and the refund has not arrived."
Reply: {"category": "billing", "reason": "Duplicate charge and refund request for an order."}

Ticket: "Someone made a transaction on my account that I did not authorise."
Reply: {"category": "fraud", "reason": "Customer reports an unauthorised transaction."}

Ticket: "The app freezes on the payment page every time I try to check out."
Reply: {"category": "technical", "reason": "App malfunction during checkout."}
"""

# Verify the requirements
for name, prompt in [("zero-shot", ZERO_SHOT_PROMPT), ("few-shot", FEW_SHOT_PROMPT)]:
    assert "CLASSIFY" in prompt, f"{name}: missing CLASSIFY"
    for cat in ["billing", "delivery", "technical", "fraud"]:
        assert cat in prompt, f"{name}: missing category {cat}"
    assert '{"category": "...", "reason": "..."}' in prompt, f"{name}: missing JSON format"

assert "EXAMPLES:" not in ZERO_SHOT_PROMPT, "zero-shot must not contain EXAMPLES:"
assert "EXAMPLES:" in FEW_SHOT_PROMPT.splitlines(), "few-shot needs an EXAMPLES: line"
assert FEW_SHOT_PROMPT.count("Ticket:") >= 2, "few-shot needs at least two examples"
assert '"category": "fraud"' in FEW_SHOT_PROMPT, "few-shot needs a fraud example"
print("Both prompts pass the checks.")

Both prompts pass the checks.


In [67]:
import logging
import random
import time

import pandas as pd
import mock_llm
from mock_llm import TransientLLMError

BASE_DELAY = 0.05   # seconds
MAX_RETRIES = 3

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("triage")

retry_log = []      # one entry per retry, so we can count them

def call_model(system_prompt, user_message):
    """Real model call through the mock's chat() API. Returns the raw reply text."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},   # redacted text only
    ]
    response = mock_llm.chat(messages)
    return response.choices[0].message.content

def call_with_retry(ticket_id, call, *args, **kwargs):
    """Retry only TransientLLMError, up to MAX_RETRIES times, with exponential backoff and jitter."""
    retries_done = 0
    while True:
        try:
            return call(*args, **kwargs)
        except TransientLLMError:
            if retries_done >= MAX_RETRIES:
                raise   # caller records a permanent error
            wait = BASE_DELAY * (2 ** retries_done) * random.uniform(0.9, 1.15)
            log.info(f"ticket={ticket_id} retry={retries_done} wait={wait:.3f}s")
            retry_log.append(ticket_id)
            time.sleep(wait)
            retries_done += 1

# ---- Full run over the 156 passed tickets, in file order ----
mock_llm.reset()
retry_log.clear()

tickets["guard_status"] = guard_df["guard_status"].values
results = []

for _, row in tickets.iterrows():
    if row["guard_status"] != "passed":
        results.append({"ticket_id": row["ticket_id"], "status": row["guard_status"]})
        continue
    try:
        reply = call_with_retry(row["ticket_id"], call_model,
                                FEW_SHOT_PROMPT, row["text_redacted"])
        results.append({"ticket_id": row["ticket_id"], "status": "ok", "reply": reply})
    except TransientLLMError:
        results.append({"ticket_id": row["ticket_id"], "status": "permanent_error"})

results_df = pd.DataFrame(results)
print(results_df["status"].value_counts())
print("Retries (TransientLLMErrors recovered):", len(retry_log))
print("Total mock calls:", mock_llm._state["calls"])

ticket=T0002 retry=0 wait=0.045s
ticket=T0116 retry=0 wait=0.055s
ticket=T0153 retry=0 wait=0.051s
ticket=T0039 retry=0 wait=0.054s
ticket=T0152 retry=0 wait=0.048s
ticket=T0043 retry=0 wait=0.049s
ticket=T0111 retry=0 wait=0.053s
ticket=T0155 retry=0 wait=0.057s
ticket=T0100 retry=0 wait=0.048s
ticket=T0068 retry=0 wait=0.048s
ticket=T0135 retry=0 wait=0.055s
ticket=T0009 retry=0 wait=0.053s
ticket=T0067 retry=0 wait=0.056s
ticket=T0106 retry=0 wait=0.057s
ticket=T0097 retry=0 wait=0.055s
ticket=T0003 retry=0 wait=0.056s
ticket=T0019 retry=0 wait=0.052s
ticket=T0086 retry=0 wait=0.046s
ticket=T0065 retry=0 wait=0.052s


status
ok               156
skipped_empty      4
flagged_long       4
Name: count, dtype: int64
Retries (TransientLLMErrors recovered): 19
Total mock calls: 175


Every retry is retry=0 because each simulated failure succeeds on the first retry, so the backoff never escalates past the first step. The ok status only means the call didn't fail. It doesn't mean the reply is usable. The mock still returns non-JSON replies and out-of-list categories like shipping, so Task 5 needs to check each reply before we accept it.

In [71]:
import json
import re

ALLOWED = ("billing", "delivery", "technical", "fraud")

def call_chat(system_prompt, user_message):
    """Like call_model, but returns the full response so we can read usage.total_tokens."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},  # redacted text only
    ]
    return mock_llm.chat(messages)

def validate_reply(text):
    """Return (category, status). Never guesses: anything unclear is flagged for human review."""
    try:
        obj = json.loads(text)
        is_json = True
    except (TypeError, json.JSONDecodeError):
        obj, is_json = None, False

    # Path 1: valid JSON
    if is_json:
        if isinstance(obj, dict) and obj.get("category") in ALLOWED:
            return obj["category"], "classified_json"
        return None, "flagged_invalid_output"   # JSON, but category outside the list (or missing)

    # Path 2: not JSON -> extract lowercase words and look for exactly one allowed category word
    words = re.findall(r"[a-z]+", (text or "").lower())
    hits = [w for w in words if w in ALLOWED]
    if len(hits) == 1:
        return hits[0], "classified_keyword"
    return None, "flagged_invalid_output"       # zero or several category words: ambiguous

In [72]:
mock_llm.reset()
retry_log.clear()

statuses = []
for _, row in tickets.iterrows():
    if row["guard_status"] != "passed":
        continue
    try:
        resp = call_with_retry(row["ticket_id"], call_chat, FEW_SHOT_PROMPT, row["text_redacted"])
    except TransientLLMError:
        statuses.append("permanent_error")
        continue
    category, status = validate_reply(resp.choices[0].message.content)
    statuses.append(status)

print(pd.Series(statuses).value_counts())

ticket=T0002 retry=0 wait=0.049s
ticket=T0116 retry=0 wait=0.047s


ticket=T0153 retry=0 wait=0.045s
ticket=T0039 retry=0 wait=0.049s
ticket=T0152 retry=0 wait=0.051s
ticket=T0043 retry=0 wait=0.046s
ticket=T0111 retry=0 wait=0.050s
ticket=T0155 retry=0 wait=0.054s
ticket=T0100 retry=0 wait=0.049s
ticket=T0068 retry=0 wait=0.046s
ticket=T0135 retry=0 wait=0.048s
ticket=T0009 retry=0 wait=0.056s
ticket=T0067 retry=0 wait=0.045s
ticket=T0106 retry=0 wait=0.048s
ticket=T0097 retry=0 wait=0.055s
ticket=T0003 retry=0 wait=0.051s
ticket=T0019 retry=0 wait=0.045s
ticket=T0086 retry=0 wait=0.046s
ticket=T0065 retry=0 wait=0.052s


classified_json           135
classified_keyword         12
flagged_invalid_output      9
Name: count, dtype: int64


In [73]:
import json
import logging
import re

import pandas as pd
import mock_llm
from mock_llm import TransientLLMError

logging.getLogger("triage").setLevel(logging.WARNING)  # hide per-retry lines in this comparison

ALLOWED = ["billing", "delivery", "technical", "fraud"]
WORD_RE = re.compile(r"\b(billing|delivery|technical|fraud)\b", re.I)

def parse_reply(content):
    """Return (category, method). Method is classified_json, classified_keyword, or unclassified."""
    try:
        obj = json.loads(content)
        if isinstance(obj, dict) and obj.get("category") in ALLOWED:
            return obj["category"], "classified_json"
    except (TypeError, json.JSONDecodeError):
        pass
    m = WORD_RE.search(content or "")
    if m:
        return m.group(1).lower(), "classified_keyword"
    return None, "unclassified"

def call_chat(system_prompt, user_message):
    """Like call_model, but returns the full response so we can read usage.total_tokens."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},  # redacted text only
    ]
    return mock_llm.chat(messages)

def run_pipeline(prompt):
    mock_llm.reset()
    rows, total_tokens = [], 0
    for _, row in tickets.iterrows():
        if row["guard_status"] != "passed":
            continue
        try:
            resp = call_with_retry(row["ticket_id"], call_chat, prompt, row["text_redacted"])
        except TransientLLMError:
            rows.append({"ticket_id": row["ticket_id"], "true": row["true_category"],
                         "pred": None, "method": "permanent_error"})
            continue
        total_tokens += resp.usage.total_tokens
        pred, method = validate_reply(resp.choices[0].message.content)
        rows.append({"ticket_id": row["ticket_id"], "true": row["true_category"],
                     "pred": pred, "method": method})
    df = pd.DataFrame(rows)
    classified = df[df["pred"].notna()]
    accuracy = (classified["pred"] == classified["true"]).mean()
    return df, classified, accuracy, total_tokens

def summarise(name, df, classified, accuracy, total_tokens):
    n = len(classified)
    print(f"--- {name} ---")
    print("Method counts:", df["method"].value_counts().to_dict())
    print(f"Classified tickets: {n}")
    print(f"Accuracy: {accuracy:.1%} ({(classified['pred'] == classified['true']).sum()} of {n})")
    print(f"Total tokens: {total_tokens}")
    print(f"Tokens per classified ticket: {total_tokens / n:.1f}\n")

zs_df, zs_cls, zs_acc, zs_tok = run_pipeline(ZERO_SHOT_PROMPT)
summarise("Zero-shot", zs_df, zs_cls, zs_acc, zs_tok)

fs_df, fs_cls, fs_acc, fs_tok = run_pipeline(FEW_SHOT_PROMPT)
summarise("Few-shot", fs_df, fs_cls, fs_acc, fs_tok)

# Confusion table for the zero-shot run (rows = true, columns = predicted)
print("Zero-shot confusion table:")
print(pd.crosstab(zs_cls["true"], zs_cls["pred"], rownames=["true"], colnames=["predicted"]))

--- Zero-shot ---
Method counts: {'classified_json': 135, 'classified_keyword': 12, 'flagged_invalid_output': 9}
Classified tickets: 147
Accuracy: 85.7% (126 of 147)
Total tokens: 12988
Tokens per classified ticket: 88.4

--- Few-shot ---
Method counts: {'classified_json': 135, 'classified_keyword': 12, 'flagged_invalid_output': 9}
Classified tickets: 147
Accuracy: 100.0% (147 of 147)
Total tokens: 27652
Tokens per classified ticket: 188.1

Zero-shot confusion table:
predicted  billing  delivery  fraud  technical
true                                          
billing         38         0      0          0
delivery         0        36      0          0
fraud           11         0     23          0
technical       10         0      0         29


The few-shot prompt costs about 99.7 extra tokens per ticket (188.1 vs 88.4), or 14,664 tokens over the run. That buys 21 corrected tickets, roughly 700 extra tokens per fixed error. The extra cost is worth paying when a wrong category is expensive, such as a fraud ticket routed to billing, where a real customer's fraud complaint goes to the wrong team. If you only need rough volume counts, or the errors are cheap to catch, zero-shot's roughly half-price tokens are the better choice. The mock's keyword rules change order when EXAMPLES: is present, so its 100% overstates what a real LLM would do, and you should test on a real model before drawing a firm conclusion.

In [74]:
import os
import pandas as pd
import mock_llm
from mock_llm import TransientLLMError

os.makedirs("outputs", exist_ok=True)

SKIP_STATUSES = {"skipped_empty", "flagged_long", "flagged_invalid_output",
                 "budget_exhausted", "permanent_error"}

def run_triage(prompt, budget):
    """Run every eligible ticket through the model, stopping new calls once the token budget is spent."""
    mock_llm.reset()
    retry_log.clear()
    used = 0                       # running total of usage.total_tokens
    rows = []
    for _, row in tickets.iterrows():
        base = {"ticket_id": row["ticket_id"], "customer_id": row["customer_id"]}

        # Input guard from Task 1: never sent to the model
        if row["guard_status"] != "passed":
            rows.append({**base, "status": row["guard_status"],
                         "category": None, "tokens_used": 0})
            continue

        # Budget check before each model call (no call is made if exhausted)
        if used >= budget:
            rows.append({**base, "status": "budget_exhausted",
                         "category": None, "tokens_used": 0})
            continue

        try:
            resp = call_with_retry(row["ticket_id"], call_chat, prompt, row["text_redacted"])
        except TransientLLMError:
            rows.append({**base, "status": "permanent_error",
                         "category": None, "tokens_used": 0})
            continue

        tok = resp.usage.total_tokens
        used += tok
        category, status = validate_reply(resp.choices[0].message.content)
        rows.append({**base, "status": status, "category": category, "tokens_used": tok})
    return pd.DataFrame(rows), used

# (a) Budget demo: zero-shot with a small budget
demo_df, demo_used = run_triage(ZERO_SHOT_PROMPT, budget=6000)
demo_df.to_csv("outputs/triage_budget_demo.csv", index=False)
print("Demo tokens used:", demo_used)
print(demo_df["status"].value_counts())
print("budget_exhausted count:", (demo_df["status"] == "budget_exhausted").sum())

# (b) Final few-shot run with a large budget
final_df, final_used = run_triage(FEW_SHOT_PROMPT, budget=100000)

# Load segments from Part B
seg = pd.read_csv("outputs/customer_segments.csv")
print("Segment file columns:", list(seg.columns))   # confirm these names
seg = seg.drop_duplicates("customer_id")
seg_name = {r.customer_id: r.segment_name for r in seg.itertuples()}
seg_premium = {r.customer_id: str(r.is_premium).strip().lower() in ("true", "1", "yes")
               for r in seg.itertuples()}

def route(row):
    """Return (route, priority) using the rules from the task. Human review is checked first."""
    if row["status"] in SKIP_STATUSES:
        return "human_review", "NORMAL"
    if row["category"] == "fraud":
        return "security_team", "HIGH"
    if seg_premium.get(row["customer_id"], False):   # no segment row = not premium
        return "premium_desk", "HIGH"
    return "support_queue", "NORMAL"

final_df["segment_name"] = final_df["customer_id"].map(seg_name)   # blank if no row
routes = final_df.apply(route, axis=1, result_type="expand")
final_df["route"] = routes[0]
final_df["priority"] = routes[1]

cols = ["ticket_id", "customer_id", "status", "category", "segment_name",
        "priority", "route", "tokens_used"]
final_df = final_df[cols]
final_df.to_csv("outputs/triage_results.csv", index=False)

print("Final tokens used:", final_used)
print(final_df["route"].value_counts())
print(final_df["priority"].value_counts())

Demo tokens used: 6004
status
budget_exhausted          84
classified_json           63
classified_keyword         5
skipped_empty              4
flagged_long               4
flagged_invalid_output     4
Name: count, dtype: int64
budget_exhausted count: 84
Segment file columns: ['customer_id', 'recency', 'frequency', 'monetary', 'segment_id', 'segment_name', 'is_premium']
Final tokens used: 27652
route
support_queue    102
security_team     34
human_review      17
premium_desk      11
Name: count, dtype: int64
priority
NORMAL    119
HIGH       45
Name: count, dtype: int64


In [75]:
import glob
import os
import pandas as pd

CHUNK_WORDS = 40
OVERLAP = 10
STEP = CHUNK_WORDS - OVERLAP   # 30: each new chunk starts 30 words later

def chunk_words(words):
    """Split a list of words into 40-word chunks with a 10-word overlap.
    Stop after the first chunk whose end reaches or passes the last word."""
    if not words:
        return []
    chunks, start, n = [], 0, len(words)
    while True:
        end = min(start + CHUNK_WORDS, n)        # exclusive end index
        chunks.append((start, end, words[start:end]))
        if start + CHUNK_WORDS - 1 >= n - 1:     # this chunk reaches the last word
            break
        start += STEP
    return chunks

# Read the five policy files and chunk each one
chunk_records = []
for path in sorted(glob.glob("data/policies/*.txt")):
    source = os.path.basename(path)
    with open(path, encoding="utf-8") as f:
        words = f.read().split()
    for i, (start, end, chunk) in enumerate(chunk_words(words)):
        chunk_records.append({
            "source": source,
            "chunk_id": f"{os.path.splitext(source)[0]}_{i}",
            "start_word": start,
            "end_word": end - 1,                 # inclusive, matches "words 0-39"
            "text": " ".join(chunk),
        })

chunks_df = pd.DataFrame(chunk_records)

print("Chunks per source:")
print(chunks_df["source"].value_counts().sort_index())
print("Total chunks:", len(chunks_df))
print(chunks_df[["source", "chunk_id", "start_word", "end_word"]].head(6))

Chunks per source:
source
delivery_policy.txt      3
escalation_policy.txt    3
payment_policy.txt       3
privacy_policy.txt       2
refund_policy.txt        4
Name: count, dtype: int64
Total chunks: 15
                  source             chunk_id  start_word  end_word
0    delivery_policy.txt    delivery_policy_0           0        39
1    delivery_policy.txt    delivery_policy_1          30        69
2    delivery_policy.txt    delivery_policy_2          60        99
3  escalation_policy.txt  escalation_policy_0           0        39
4  escalation_policy.txt  escalation_policy_1          30        69
5  escalation_policy.txt  escalation_policy_2          60        77


In [76]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Fit once on all chunk texts
vectorizer = TfidfVectorizer()
chunk_matrix = vectorizer.fit_transform(chunks_df["text"])

def retrieve(query, k=3):
    """Return the top-k chunks for the query, ranked by cosine similarity."""
    query_vec = vectorizer.transform([query])            # same fitted vocabulary
    scores = cosine_similarity(query_vec, chunk_matrix)[0]
    top_idx = np.argsort(scores)[::-1][:k]               # highest score first
    return [
        {
            "source": chunks_df.iloc[i]["source"],
            "score": float(scores[i]),
            "text": chunks_df.iloc[i]["text"],
        }
        for i in top_idx
    ]

# Check against the reference query
for hit in retrieve("refund policy late delivery", k=3):
    print(f"{hit['source']}  score={hit['score']:.3f}")

refund_policy.txt  score=0.272
refund_policy.txt  score=0.251
refund_policy.txt  score=0.245


All three top hits come from refund_policy.txt, matching the reference. The scores (0.272, 0.251, 0.245) are close together, which is typical for TF-IDF over short chunks, so the ranking is clear but the margins are small.



In [77]:
SYSTEM_POLICY = """You are a POLICY ASSISTANT for Swiggy Instamart customer support.
Answer the question using ONLY the information in the CONTEXT provided by the user.
If the CONTEXT does not contain the answer, reply exactly:
I could not find that information in the policy documents."""

def answer(question, k=3):
    """Retrieve k chunks, ask the mock policy assistant to answer from them, return text and sources."""
    hits = retrieve(question, k=k)

    # One block per chunk, each prefixed with its source file name on its own line
    context_blocks = [f"[{h['source']}]\n{h['text']}" for h in hits]
    user_message = "CONTEXT:\n" + "\n".join(context_blocks) + "\nQUESTION:\n" + question

    messages = [
        {"role": "system", "content": SYSTEM_POLICY},
        {"role": "user", "content": user_message},
    ]
    response = call_with_retry("policy_qa", mock_llm.chat, messages)

    # Unique source files, in the order retrieved
    sources = list(dict.fromkeys(h["source"] for h in hits))
    return {"answer": response.choices[0].message.content, "sources": sources}

# Quick check
result = answer("What is the refund policy for late delivery?")
print("Answer:", result["answer"])
print("Sources:", result["sources"])

Answer: Refunds up to INR 1000 are
 
Refund Policy.
Sources: ['refund_policy.txt']


This comes from fixed 40-word chunking, which doesn't respect sentence boundaries. 

In [83]:
import numpy as np
import pandas as pd
import glob
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import mock_llm

eval_df = pd.read_csv("data/raw/policy_eval.csv")
print("Columns:", eval_df.columns.tolist())   # confirm the names below
print("Rows:", len(eval_df))

def make_chunks(size, overlap):
    """Same stop rule as Task 1 of Part D, with a configurable size and overlap."""
    step = size - overlap
    records = []
    for path in sorted(glob.glob("data/policies/*.txt")):
        source = os.path.basename(path)
        with open(path, encoding="utf-8") as f:
            words = f.read().split()
        start, n, i = 0, len(words), 0
        while True:
            end = min(start + size, n)
            records.append({"source": source, "chunk_id": f"{source}_{i}",
                            "text": " ".join(words[start:end])})
            if start + size - 1 >= n - 1:
                break
            start += step
            i += 1
    return pd.DataFrame(records)

def build_index(chunks):
    """Fit TF-IDF on one chunk set; return the (chunks, vectorizer, matrix) bundle."""
    vec = TfidfVectorizer()
    matrix = vec.fit_transform(chunks["text"])
    return chunks, vec, matrix

def retrieve_from(index, query, k=3):
    chunks, vec, matrix = index
    scores = cosine_similarity(vec.transform([query]), matrix)[0]
    top = np.argsort(scores)[::-1][:k]
    return [{"source": chunks.iloc[i]["source"], "score": float(scores[i]),
             "text": chunks.iloc[i]["text"]} for i in top]

def answer_from(index, question, k=3):
    hits = retrieve_from(index, question, k=k)
    context = "\n".join(f"[{h['source']}]\n{h['text']}" for h in hits)
    messages = [
        {"role": "system", "content": SYSTEM_POLICY},
        {"role": "user", "content": "CONTEXT:\n" + context + "\nQUESTION:\n" + question},
    ]
    response = call_with_retry("policy_eval", mock_llm.chat, messages)
    return response.choices[0].message.content, [h["source"] for h in hits]

def evaluate(size, overlap):
    index = build_index(make_chunks(size, overlap))
    rows = []
    for _, r in eval_df.iterrows():
        text, sources = answer_from(index, r["question"])
        answerable = isinstance(r["expected_doc"], str) and r["expected_doc"].strip().lower().endswith(".txt")
        if answerable:
            rows.append({"question": r["question"], "answerable": True,
                         "doc_in_top3": r["expected_doc"] in sources,
                         "fact_in_answer": str(r["expected_fact"]) in text,
                         "answer": text})
        else:
            rows.append({"question": r["question"], "answerable": False,
                         "fallback": text.startswith("I could not find"),
                         "answer": text})
    return pd.DataFrame(rows)

def summarise(name, res):
    ans = res[res["answerable"]]
    unans = res[~res["answerable"]]
    print(f"--- {name} ---")
    print(f"Answerable: {len(ans)}  expected_doc in top 3: {ans['doc_in_top3'].sum()}  "
          f"expected_fact in answer: {ans['fact_in_answer'].sum()}")
    fallback_count = unans["fallback"].sum() if "fallback" in unans.columns else 0
    print(f"Unanswerable: {len(unans)}  fallback answer: {fallback_count}\n")

mock_llm.reset()
overlap10 = evaluate(40, 10)
summarise("Overlap 10 (40-word chunks)", overlap10)

mock_llm.reset()
overlap0 = evaluate(40, 0)
summarise("Overlap 0 (40-word chunks)", overlap0)

print("Failing answerable questions with overlap 0:")
print(overlap0[overlap0["answerable"] & ~(overlap0["doc_in_top3"] & overlap0["fact_in_answer"])]
      [["question", "doc_in_top3", "fact_in_answer", "answer"]].to_string())

Columns: ['question', 'expected_doc', 'expected_fact']
Rows: 14
--- Overlap 10 (40-word chunks) ---
Answerable: 12  expected_doc in top 3: 12  expected_fact in answer: 12
Unanswerable: 2  fallback answer: 2

--- Overlap 0 (40-word chunks) ---
Answerable: 12  expected_doc in top 3: 12  expected_fact in answer: 10
Unanswerable: 2  fallback answer: 2

Failing answerable questions with overlap 0:
                                                                 question doc_in_top3 fact_in_answer                                                                                                                                                      answer
0  What delivery time makes a completed order eligible for a full refund?        True          False                                                                                                                 receives a full refund of the order amount.
7        How soon is money returned when a payment fails but is deducted?        True     

In [82]:
print(eval_df["expected_doc"].value_counts(dropna=False))
print(eval_df[eval_df["expected_doc"].isna() | ~eval_df["expected_doc"].astype(str).str.endswith(".txt")])

expected_doc
refund_policy.txt        3
delivery_policy.txt      3
payment_policy.txt       3
escalation_policy.txt    2
NONE                     2
privacy_policy.txt       1
Name: count, dtype: int64
                                             question expected_doc  \
12  What is the favourite colour of the company ma...         NONE   
13               Who won the cricket world cup final?         NONE   

   expected_fact  
12           NaN  
13           NaN  


"What delivery time makes a completed order eligible for a full refund?" The answer contains "more than 45 minutes" in its sentence, but with no overlap that sentence is cut between two chunks. The top chunk holds only the start, so the mock returns "receives a full refund of the order amount.", which leaves out the time threshold.

"How soon is money returned when a payment fails but is deducted?" The relevant sentence is split the same way. The mock returns a fragment of the payment policy, then a fragment of the refund policy ("1000 must be approved by a human support lead..."), so the answer mixes two rules and misses the expected fact.

The document was still retrieved in both cases, so retrieval didn't fail. The information was present in the corpus but was split across a chunk boundary, and with overlap 0 the chunk that was retrieved didn't contain the complete sentence. The 10-word overlap gives each boundary a second chance, which is why the same questions pass there.

In [110]:
import json
import sqlite3
from datetime import datetime, timezone
import os

os.makedirs("outputs", exist_ok=True)
DB_PATH = "data/instamart.db"

# ---- Tool 1: read one order from the Part A database ----
def _num(x):
    """Convert to float, but keep NULL (None) as None instead of crashing."""
    return None if x is None else float(x)

def get_order(order_id):
    """Return the order's status, amount and delivery time, or an error if it does not exist."""
    con = sqlite3.connect(DB_PATH)
    try:
        row = con.execute(
            "SELECT order_id, status, order_amount, delivery_minutes FROM orders WHERE order_id = ?",
            (order_id,),
        ).fetchone()
    finally:
        con.close()
    if row is None:
        return {"error": "order not found"}
    return {"order_id": row[0], "status": row[1],
            "order_amount": _num(row[2]), "delivery_minutes": _num(row[3])}

# ---- Tool 2: retrieve policy chunks from Part D Task 2 ----
def search_policy(query):
    """Return the top retrieved policy chunks for the query."""
    return {"results": retrieve(query, k=3)}

# ---- Tool 3: apply the refund rule ----
def calculate_refund(order_amount, delivery_minutes):
    """Full refund if delivered after 45 minutes, otherwise zero. Order status is checked by the agent."""
    if float(delivery_minutes) > 45:
        return {"refund_inr": float(order_amount)}
    return {"refund_inr": 0.0}

# ---- Helper: append one line to the refund ledger ----
def _ledger(tool_name, **fields):
    entry = {"timestamp": datetime.now(timezone.utc).isoformat(), "tool": tool_name, **fields}
    with open(LEDGER_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")

# ---- Tool 4: issue a refund automatically ----
def issue_refund(order_id, refund_inr):
    _ledger("issue_refund", order_id=order_id, refund_inr=float(refund_inr))
    return {"refund_inr": float(refund_inr)}

# ---- Tool 5: send a refund to a human lead ----
def escalate_to_human(order_id, refund_inr):
    _ledger("escalate_to_human", order_id=order_id, refund_inr=float(refund_inr))
    return {"refund_inr": float(refund_inr)}

# Map from tool name (what the model asks for) to the Python function
TOOL_MAP = {
    "get_order": get_order,
    "search_policy": search_policy,
    "calculate_refund": calculate_refund,
    "issue_refund": issue_refund,
    "escalate_to_human": escalate_to_human,
}

# JSON-style specs describing each tool to the model
TOOLS_SPEC = [
    {"type": "function", "function": {
        "name": "get_order",
        "description": "Look up an order's status, order amount and delivery time by order id.",
        "parameters": {"type": "object",
                       "properties": {"order_id": {"type": "string"}},
                       "required": ["order_id"]}}},
    {"type": "function", "function": {
        "name": "search_policy",
        "description": "Search the company policy documents and return the most relevant passages.",
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string"}},
                       "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "calculate_refund",
        "description": "Compute the refund amount from the order amount and delivery time.",
        "parameters": {"type": "object",
                       "properties": {"order_amount": {"type": "number"},
                                      "delivery_minutes": {"type": "number"}},
                       "required": ["order_amount", "delivery_minutes"]}}},
    {"type": "function", "function": {
        "name": "issue_refund",
        "description": "Issue a refund automatically and record it in the refund ledger.",
        "parameters": {"type": "object",
                       "properties": {"order_id": {"type": "string"},
                                      "refund_inr": {"type": "number"}},
                       "required": ["order_id", "refund_inr"]}}},
    {"type": "function", "function": {
        "name": "escalate_to_human",
        "description": "Send a refund above INR 1000 to a human support lead and record it in the ledger.",
        "parameters": {"type": "object",
                       "properties": {"order_id": {"type": "string"},
                                      "refund_inr": {"type": "number"}},
                       "required": ["order_id", "refund_inr"]}}},
]

# ---- Quick checks ----
con = sqlite3.connect(DB_PATH)
print("Tables:", [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")])
con.close()

print(get_order("ORD-000000"))                      # expected: {"error": "order not found"}
print(calculate_refund(500, 50))                    # expected: refund 500.0
print(calculate_refund(500, 30))                    # expected: refund 0.0
print(sorted(TOOL_MAP.keys()))

Tables: ['customers', 'products', 'orders', 'tickets']
{'error': 'order not found'}
{'refund_inr': 500.0}
{'refund_inr': 0.0}
['calculate_refund', 'escalate_to_human', 'get_order', 'issue_refund', 'search_policy']


In [109]:
import json
import time
import pandas as pd
import mock_llm

SYSTEM_AGENT = (
    "You are a refund AGENT for Swiggy Instamart. Use the tools to look up the order, "
    "search the policy, calculate the refund, then either issue it or escalate it. "
    "Finish with one line starting with REFUND:, ESCALATE:, DENIED: or NOT_FOUND:."
)

def run_agent(message, scenario_id, max_iterations=6,
              token_limit=6000, time_limit_seconds=10):
    """ReAct loop: call the model, run any requested tools, feed results back, stop at a final answer."""
    messages = [
        {"role": "system", "content": SYSTEM_AGENT},
        {"role": "user", "content": message},
    ]
    steps, llm_calls, tokens_used = [], 0, 0
    final_answer, status = None, None
    start = time.time()

    for _ in range(max_iterations):
        if time.time() - start > time_limit_seconds:
            status = "TIME_LIMIT"
            break
        if tokens_used >= token_limit:
            status = "TOKEN_LIMIT"
            break

        resp = call_with_retry(scenario_id, mock_llm.chat, messages, tools=TOOLS_SPEC)
        llm_calls += 1
        tokens_used += resp.usage.total_tokens
        msg = resp.choices[0].message

        # No tool calls means this reply is the final answer
        if not msg.tool_calls:
            final_answer = msg.content
            break

        tool_calls = [{"id": tc.id, "name": tc.name, "arguments": tc.arguments}
                      for tc in msg.tool_calls]
        messages.append({"role": "assistant", "content": None, "tool_calls": tool_calls})

        for tc in tool_calls:
            fn = TOOL_MAP.get(tc["name"])
            result = fn(**tc["arguments"]) if fn else {"error": f"unknown tool {tc['name']}"}
            messages.append({"role": "tool", "tool_call_id": tc["id"], "name": tc["name"],
                             "content": json.dumps(result)})
            steps.append({"tool": tc["name"], "arguments": tc["arguments"], "observation": result})

    if final_answer is not None:
        status = final_answer.split(":")[0].strip()   # word before the colon
    elif status is None:
        status = "MAX_ITERATIONS"                     # loop ended without a final answer

    # Refund amount = refund_inr from the last calculate_refund step, if any
    calc = [s["observation"]["refund_inr"] for s in steps if s["tool"] == "calculate_refund"]
    refund_inr = calc[-1] if calc else None

    return {"scenario_id": scenario_id, "status": status, "final_answer": final_answer,
            "refund_inr": refund_inr, "llm_calls": llm_calls,
            "tokens_used": tokens_used, "steps": steps}

# ---- Run all 7 scenarios and write one JSON line each ----
scenarios = pd.read_csv("data/raw/agent_scenarios.csv")

results = []
with open("outputs/agent_runs.jsonl", "w", encoding="utf-8") as f:   # fresh file for this batch
    for _, row in scenarios.iterrows():
        res = run_agent(row["message"], row["scenario_id"])
        results.append(res)
        f.write(json.dumps({k: res[k] for k in
                            ["scenario_id", "status", "final_answer", "llm_calls",
                             "tokens_used", "steps"]}) + "\n")

summary = pd.DataFrame([{"scenario_id": r["scenario_id"], "status": r["status"],
                         "refund_inr": r["refund_inr"], "llm_calls": r["llm_calls"],
                         "tokens_used": r["tokens_used"]} for r in results])
print(summary.to_string(index=False))

# Check the refund amounts against the expected values
expected = scenarios.set_index("scenario_id")["expected_refund_inr"]
for r in results:
    exp = expected[r["scenario_id"]]
    got = r["refund_inr"] if r["refund_inr"] is not None else 0.0
    print(r["scenario_id"], "OK" if abs(got - exp) <= 0.01 else f"MISMATCH expected {exp}, got {got}")

scenario_id         status  refund_inr  llm_calls  tokens_used
         S1         REFUND      128.25          5         1092
         S2         REFUND      135.00          5         1092
         S3       ESCALATE     1999.00          5         1079
         S4         DENIED        0.00          4          761
         S5         DENIED         NaN          3          454
         S6      NOT_FOUND         NaN          2          150
         S7 MAX_ITERATIONS         NaN          6         3234
S1 OK
S2 OK
S3 OK
S4 OK
S5 OK
S6 OK
S7 OK


The Part D section needs the seven-row table with scenario id, status, refund amount, model calls and tokens used. You can copy it straight from the summary output.


In [117]:
import json
import time
import pandas as pd
import mock_llm

SYSTEM_AGENT = (
    "You are a refund AGENT for Swiggy Instamart. Use the tools to look up the order, "
    "search the policy, calculate the refund, then either issue it or escalate it. "
    "Finish with one line starting with REFUND:, ESCALATE:, DENIED: or NOT_FOUND:."
)

def run_agent(user_message, scenario_id, max_iterations=6,
              token_limit=6000, time_limit_seconds=10):
    """ReAct loop with three stop limits and a graceful human-handoff fallback."""
    messages = [
        {"role": "system", "content": SYSTEM_AGENT},
        {"role": "user", "content": user_message},
    ]
    steps, llm_calls, tokens_used = [], 0, 0
    final_answer, status = None, None
    start = time.monotonic()                      # monotonic clock, unaffected by system time changes

    def handoff(reason):
        """Short summary for a human agent taking over an unfinished run."""
        last_tool = steps[-1]["tool"] if steps else "none"
        return (f"HANDOFF: {reason}. {llm_calls} model call(s), {len(steps)} tool step(s), "
                f"last tool: {last_tool}. Please review this request manually.")

    for _ in range(max_iterations):
        # (c) Wall-clock limit, checked at the top of each iteration
        if time.monotonic() - start >= time_limit_seconds:
            status = "FALLBACK_TIMEOUT"
            final_answer = handoff("time limit reached")
            break

        resp = call_with_retry(scenario_id, mock_llm.chat, messages, tools=TOOLS_SPEC)
        llm_calls += 1
        tokens_used += resp.usage.total_tokens

        # (b) Token budget, checked before any tool from this reply is executed
        if tokens_used > token_limit:
            status = "FALLBACK_TOKEN_LIMIT"
            final_answer = handoff("token limit exceeded")
            break

        msg = resp.choices[0].message

        # No tool calls means this reply is the final answer
        if not msg.tool_calls:
            final_answer = msg.content
            break

        tool_calls = [{"id": tc.id, "name": tc.name, "arguments": tc.arguments}
                      for tc in msg.tool_calls]
        messages.append({"role": "assistant", "content": None, "tool_calls": tool_calls})

        for tc in tool_calls:
            fn = TOOL_MAP.get(tc["name"])
            result = fn(**tc["arguments"]) if fn else {"error": f"unknown tool {tc['name']}"}
            messages.append({"role": "tool", "tool_call_id": tc["id"], "name": tc["name"],
                             "content": json.dumps(result)})
            steps.append({"tool": tc["name"], "arguments": tc["arguments"], "observation": result})
    else:
        # (a) Loop used all max_iterations model calls without a final answer
        status = "FALLBACK_MAX_ITERATIONS"
        final_answer = handoff("iteration limit reached without a final answer")

    if status is None:
        status = final_answer.split(":")[0].strip()   # word before the colon

    calc = [s["observation"]["refund_inr"] for s in steps if s["tool"] == "calculate_refund"]
    refund_inr = calc[-1] if calc else None

    return {"scenario_id": scenario_id, "status": status, "final_answer": final_answer,
            "refund_inr": refund_inr, "llm_calls": llm_calls,
            "tokens_used": tokens_used, "steps": steps}

# ---- Demo 1: token limit (S1 with token_limit=200) ----
s1 = scenarios[scenarios["scenario_id"] == "S1"].iloc[0]
demo_tokens = run_agent(s1["message"], "S1", token_limit=200)
print("TOKEN DEMO:", demo_tokens["status"], "| calls:", demo_tokens["llm_calls"],
      "| tokens:", demo_tokens["tokens_used"], "| tool steps:", len(demo_tokens["steps"]))
print("  ", demo_tokens["final_answer"])

# ---- Demo 2: time limit (time_limit_seconds=0 stops before the first call) ----
demo_time = run_agent(s1["message"], "S1", time_limit_seconds=0)
print("TIME DEMO:", demo_time["status"], "| calls:", demo_time["llm_calls"],
      "| tokens:", demo_time["tokens_used"], "| tool steps:", len(demo_time["steps"]))
print("  ", demo_time["final_answer"])

# ---- Batch: all 7 scenarios with default limits ----
results = []
with open("outputs/agent_runs.jsonl", "w", encoding="utf-8") as f:
    for _, row in scenarios.iterrows():
        res = run_agent(row["message"], row["scenario_id"])
        results.append(res)
        f.write(json.dumps({k: res[k] for k in
                            ["scenario_id", "status", "final_answer", "llm_calls",
                             "tokens_used", "steps"]}) + "\n")

summary = pd.DataFrame([{"scenario_id": r["scenario_id"], "status": r["status"],
                         "refund_inr": r["refund_inr"], "llm_calls": r["llm_calls"],
                         "tokens_used": r["tokens_used"]} for r in results])
print(summary.to_string(index=False))

# Check refund amounts against the expected values
expected = scenarios.set_index("scenario_id")["expected_refund_inr"]
for r in results:
    exp = expected[r["scenario_id"]]
    got = r["refund_inr"] if r["refund_inr"] is not None else 0.0
    print(r["scenario_id"], "OK" if abs(got - exp) <= 0.01 else f"MISMATCH expected {exp}, got {got}")

TOKEN DEMO: FALLBACK_TOKEN_LIMIT | calls: 3 | tokens: 471 | tool steps: 2
   HANDOFF: token limit exceeded. 3 model call(s), 2 tool step(s), last tool: search_policy. Please review this request manually.
TIME DEMO: FALLBACK_TIMEOUT | calls: 0 | tokens: 0 | tool steps: 0
   HANDOFF: time limit reached. 0 model call(s), 0 tool step(s), last tool: none. Please review this request manually.
scenario_id                  status  refund_inr  llm_calls  tokens_used
         S1                  REFUND      128.25          5         1092
         S2                  REFUND      135.00          5         1092
         S3                ESCALATE     1999.00          5         1079
         S4                  DENIED        0.00          4          761
         S5                  DENIED         NaN          3          454
         S6               NOT_FOUND         NaN          2          150
         S7 FALLBACK_MAX_ITERATIONS         NaN          6         3234
S1 OK
S2 OK
S3 OK
S4 OK
S5 OK
S6 O

Part D section, record the two demo statuses alongside the scenario table:

Demo	Setting	Status	Model calls	Tokens used
S1 token limit	token_limit=200	FALLBACK_TOKEN_LIMIT	3	471
S1 time limit	time_limit_seconds=0	FALLBACK_TIMEOUT	0	0